# NACC NP - Model Development
**Target:** NPADNC Binary (0=Not/Low AD, 1=Intermediate/High AD)
**Data:** OSS-20B extraction - 161 reports

---
**Flow:** Setup -> Feature Selection -> Primary Pipeline -> Tuning -> Residual Pipeline -> Combined Pipeline -> Tuning -> Comparison -> Ensemble -> Final Fit -> SHAP -> Error Analysis

**Entering vs Retained:**
- **Entering** = features surviving missingness, NZV, one-hot, p-value, and bootstrap stability selection - the candidate pool handed to a model.
- **Retained** = entering features a fitted model actually uses (non-zero coefficient for LR, non-zero `feature_importances_` for tree models), computed per model per fold after fitting.

Primary and residual each produce their own entering set independently. Combined = primary entering + residual entering, fit fresh - its retained set is not constrained by primary's.

## 1. Setup & Data Loading

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import time
import json
import pickle
import hashlib
import inspect
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats

from sklearn.base            import BaseEstimator, TransformerMixin
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, VotingClassifier
from sklearn.svm             import SVC
from sklearn.pipeline        import Pipeline
from sklearn.compose         import ColumnTransformer
from sklearn.preprocessing   import StandardScaler, RobustScaler
from sklearn.impute          import SimpleImputer
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, cross_val_predict, cross_val_score
from sklearn.utils           import resample

from sklearn.metrics         import (
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score, confusion_matrix, ConfusionMatrixDisplay, classification_report,
    f1_score, precision_score, brier_score_loss, balanced_accuracy_score, matthews_corrcoef
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

from lightgbm import LGBMClassifier
from xgboost  import XGBClassifier

import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from joblib import Parallel, delayed
from tqdm.auto import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_PATH     = '/N/project/ADRD/neuropathoroot/results/eda_outputs/dataset_raw_merged.csv'
OUTPUT_DIR    = '/N/project/ADRD/neuropathoroot/results/model_outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Plot style ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':    'DejaVu Sans',
    'font.size':      11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.dpi':     600,
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

C_OSS   = '#12436D'
C_QWEN  = '#801650'
C_LLAMA = '#28A197'
C_GREY  = '#505a5f'
C_LIGHT = '#b1b4b6'

MODEL_COLORS = {
    'Logistic Regression': C_OSS,
    'Random Forest':       C_QWEN,
    'SVM':                 '#A8BD3A',
    'LightGBM':            '#F46A25',
    'XGBoost':             C_LLAMA,
    'Ensemble':            C_GREY,
}

def save_fig(path_no_ext):
    plt.savefig(f'{path_no_ext}.png', dpi=600, bbox_inches='tight')
    plt.savefig(f'{path_no_ext}.svg', format='svg', bbox_inches='tight')

print(f"Output directory: {OUTPUT_DIR}")

In [2]:
FEATURE_VARS = {
    # Specimen Info
    'NPSEX':     'BINARY',
    'NACCDAGE':  'CONTINUOUS',
    'NPWBRWT':   'CONTINUOUS',
    # Gross Exam
    'NPGRCCA':   'ORDINAL',
    'NPGRLA':    'BINARY',
    'NPGRHA':    'ORDINAL',
    'NPGRSNH':   'ORDINAL',
    'NPGRLCH':   'ORDINAL',
    'NACCAVAS':  'ORDINAL',
    'NPWMR':     'ORDINAL',
    'NACCARTE':  'ORDINAL',
    # AD Pathology
    'NACCAMY':   'ORDINAL',
    # Lewy Body
    'NPLBOD':    ('ONEHOT', 0),
    # Microscopic
    'NPNLOSS':   'ORDINAL',
    'NPHIPSCL':  ('ONEHOT', 0),
    'NPSCL':     'BINARY',
    # Infarcts
    'NPLINF':    'BINARY',
    'NPLAC':     'BINARY',
    'NPINF':     'BINARY',
    'NPINF1A':   'CONTINUOUS',
    'NPINF2A':   'CONTINUOUS',
    'NPINF3A':   'CONTINUOUS',
    'NPINF4A':   'CONTINUOUS',
    # Hemorrhage
    'NPHEMO':    'BINARY',
    'NPHEMO1':   'BINARY',
    'NPHEMO2':   'BINARY',
    'NPHEMO3':   'BINARY',
    # Microinfarcts
    'NPOLD':     'BINARY',
    'NPOLD1':    'ORDINAL',
    'NPOLD2':    'ORDINAL',
    'NPOLD3':    'ORDINAL',
    'NPOLD4':    'ORDINAL',
    # Microbleeds
    'NPOLDD':    'BINARY',
    'NPOLDD1':   'ORDINAL',
    'NPOLDD2':   'ORDINAL',
    'NPOLDD3':   'ORDINAL',
    'NPOLDD4':   'ORDINAL',
    # Other Vascular
    'NPMICRO':   'BINARY',
    'NPART':     'BINARY',
    'NPOANG':    'BINARY',
    'NPPATH':    'BINARY',
    'NACCNEC':   'BINARY',
    'NPPATH7':   'BINARY',
    'NPPATH8':   'BINARY',
    'NPPATH9':   'BINARY',
    'NPPATH10':  'BINARY',
    'NPPATH11':  'BINARY',
    'NPPATHO':   'BINARY',
    # FTLD Tau
    'NACCPICK':  'BINARY',
    'NPFTDT2':   'BINARY',
    'NACCCBD':   'BINARY',
    'NACCPROG':  'BINARY',
    'NPFTDT5':   'BINARY',
    'NPFTDT6':   'BINARY',
    'NPFTDT7':   'BINARY',
    'NPFTDT8':   'BINARY',
    'NPFTDT9':   'BINARY',
    'NPFTDT10':  'BINARY',
    # FTLD General
    'NPFTDTDP':  'BINARY',
    'NPALSMND':  ('ONEHOT', 0),
    'NPOFTD1':   'BINARY',
    'NPOFTD2':   'BINARY',
    'NPOFTD3':   'BINARY',
    'NPOFTD4':   'BINARY',
    'NPOFTD5':   'BINARY',
    'NPFTDNO':   'BINARY',
    'NPFTDSPC':  'BINARY',
    'NPFTD':     ('ONEHOT', 3),
    'NPTAU':     'BINARY',
    'NPFRONT':   'BINARY',
    # TDP-43
    'NPTDPA':    'BINARY',
    'NPTDPB':    'BINARY',
    'NPTDPC':    'BINARY',
    'NPTDPD':    'BINARY',
    'NPTDPE':    'BINARY',
    # Diagnoses
    'NPPLEWY':   'BINARY',
    'NPCLEWY':   'BINARY',
    'NPPVASC':   'BINARY',
    'NPCVASC':   'BINARY',
    'NPPFTLD':   'BINARY',
    'NPCFTLD':   'BINARY',
    'NPPHIPP':   'BINARY',
    'NPCHIPP':   'BINARY',
    'NPPPRION':  'BINARY',
    'NPCPRION':  'BINARY',
    'NPPOTH1':   'BINARY',
    'NPCOTH1':   'BINARY',
    'NPPOTH2':   'BINARY',
    'NPCOTH2':   'BINARY',
    'NPPOTH3':   'BINARY',
    'NPCOTH3':   'BINARY',
    'NACCOTHP':  'BINARY',
    'NACCPRIO':  'BINARY',
    # Genetics
    'NPPDXP':    'BINARY',
    'NPPDXQ':    'BINARY',
    # Other Disease
    'NPPDXA':    'BINARY',
    'NPPDXB':    'BINARY',
    'NPPDXD':    'BINARY',
    'NPPDXE':    'BINARY',
    'NPPDXF':    'BINARY',
    'NPPDXG':    'BINARY',
    'NPPDXH':    'BINARY',
    'NPPDXI':    'BINARY',
    'NPPDXJ':    'BINARY',
    'NPPDXK':    'BINARY',
    'NPPDXL':    'BINARY',
    'NPPDXM':    'BINARY',
    'NPPDXN':    'BINARY',
    # Criteria & Misc
    'NPVOTH':    'BINARY',

    # Residual: Hemibrain Weights
    'hemibrain_weight_right_fresh_g':  'CONTINUOUS',
    'hemibrain_weight_left_fresh_g':   'CONTINUOUS',
    'hemibrain_weight_right_fixed_g':  'CONTINUOUS',
    'hemibrain_weight_left_fixed_g':   'CONTINUOUS',
    # Residual: Cerebral Weights
    'cerebral_weight_right_fresh_g':   'CONTINUOUS',
    'cerebral_weight_left_fresh_g':    'CONTINUOUS',
    'cerebral_weight_right_fixed_g':   'CONTINUOUS',
    'cerebral_weight_left_fixed_g':    'CONTINUOUS',
    # Residual: Cerebellar Weights
    'cerebellar_weight_right_fresh_g': 'CONTINUOUS',
    'cerebellar_weight_left_fresh_g':  'CONTINUOUS',
    'cerebellar_weight_right_fixed_g': 'CONTINUOUS',
    'cerebellar_weight_left_fixed_g':  'CONTINUOUS',
    # Residual: Brainstem Weights
    'brainstem_weight_right_fresh_g':  'CONTINUOUS',
    'brainstem_weight_left_fresh_g':   'CONTINUOUS',
    'brainstem_weight_right_fixed_g':  'CONTINUOUS',
    'brainstem_weight_left_fixed_g':   'CONTINUOUS',
    # Residual: Weight Deltas
    'hemibrain_weight_delta_g':        'CONTINUOUS',
    'cerebral_weight_delta_g':         'CONTINUOUS',
    'cerebellar_weight_delta_g':       'CONTINUOUS',
    'brainstem_weight_delta_g':        'CONTINUOUS',
    # Residual: Corpus Callosum
    'corpus_callosum_genu_mm':              'CONTINUOUS',
    'corpus_callosum_body_anterior_mm':     'CONTINUOUS',
    'corpus_callosum_body_mid_mm':          'CONTINUOUS',
    'corpus_callosum_body_posterior_mm':    'CONTINUOUS',
    'corpus_callosum_splenium_mm':          'CONTINUOUS',
    # Residual: Circle of Willis
    'cow_basilar_mm':        'CONTINUOUS',
    'cow_vertebral_right_mm':'CONTINUOUS',
    'cow_vertebral_left_mm': 'CONTINUOUS',
    'cow_ica_right_mm':      'CONTINUOUS',
    'cow_ica_left_mm':       'CONTINUOUS',
    'cow_mca_right_mm':      'CONTINUOUS',
    'cow_mca_left_mm':       'CONTINUOUS',
    'cow_aca_right_mm':      'CONTINUOUS',
    'cow_aca_left_mm':       'CONTINUOUS',
    'cow_pca_right_mm':      'CONTINUOUS',
    'cow_pca_left_mm':       'CONTINUOUS',
    'cow_pcom_right_mm':     'CONTINUOUS',
    'cow_pcom_left_mm':      'CONTINUOUS',
    'cow_acom_mm':           'CONTINUOUS',
    # Residual: Structural Measurements
    'caudate_nucleus_head_width_mm':        'CONTINUOUS',
    # Residual: Structural Severity
    'lateral_ventricle_enlargement_severity': 'ORDINAL',
    'amygdala_atrophy_severity':              'ORDINAL',
    'dilated_perivascular_spaces':            'ORDINAL',
    'cerebellar_atrophy_severity':            'ORDINAL',
    'globus_pallidus_neuronal_loss_severity': 'ORDINAL',
    'basal_ganglia_atrophy':                  'ORDINAL',
    'thalamic_degeneration_severity':         'ORDINAL',
    'brainstem_atrophy_severity':             'ORDINAL',
    # Residual: Regional Pathology
    'purkinje_cell_loss_severity':            'ORDINAL',
    'dentate_nucleus_atrophy':                'ORDINAL',
    'artag_severity':                         'ORDINAL',
    'gvd_hippocampus_severity':               'ORDINAL',
    'hirano_bodies_hippocampus_severity':     'ORDINAL',
}

In [3]:
# ── Load raw merged dataset (pre-EDA-filtering) ────────────────────────────────
df = pd.read_csv(DATA_PATH)

TARGET_BIN = 'NPADNC_bin'

# NACC primary variables are ALL CAPS, including multi-category vars (NPLBOD,
# NPFTD) that get one-hot encoded inside FeatureSelector. Residual variables
# are lowercase snake_case.
def is_residual_var(col):
    base = col.rsplit('_', 1)[0] if col[-1].isdigit() else col
    return not base.isupper()

ONEHOT_VARS = [v for v, t in FEATURE_VARS.items() if isinstance(t, tuple) and t[0] == 'ONEHOT']
ONEHOT_REF  = {v: t[1] for v, t in FEATURE_VARS.items() if isinstance(t, tuple) and t[0] == 'ONEHOT'}

def get_var_type(var_name):
    t = FEATURE_VARS[var_name]
    return t[0] if isinstance(t, tuple) else t

ALL_RAW_VARS = [c for c in FEATURE_VARS if c in df.columns]
missing_from_csv = [c for c in FEATURE_VARS if c not in df.columns]
if missing_from_csv:
    print(f"WARNING: {len(missing_from_csv)} FEATURE_VARS not found in CSV: {missing_from_csv}")

PRIMARY_VARS  = [c for c in ALL_RAW_VARS if not is_residual_var(c)]
RESIDUAL_VARS = [c for c in ALL_RAW_VARS if is_residual_var(c)]

X_all = df[ALL_RAW_VARS].copy()
y     = df[TARGET_BIN].astype(int).copy()

# Fingerprints the underlying data so feature-selection caches (Sections 4, 6)
# auto-invalidate if the CSV changes, independent of selection parameters.
DATA_HASH = hashlib.md5(pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()

print(f"Dataset shape:     {df.shape}")
print(f"Candidate features (pre-selection): {len(ALL_RAW_VARS)}")
print(f"  Primary:  {len(PRIMARY_VARS)}")
print(f"  Residual: {len(RESIDUAL_VARS)}")
print(f"  One-hot (multi-category, primary): {ONEHOT_VARS}")
print(f"Target class 0:    {(y==0).sum()} ({(y==0).mean()*100:.1f}%)")
print(f"Target class 1:    {(y==1).sum()} ({(y==1).mean()*100:.1f}%)")
print(f"Data fingerprint:  {DATA_HASH[:12]}...")

## 2. Model Configuration

In [4]:
USE_SVM      = False
USE_LIGHTGBM = False
USE_XGBOOST  = True

# Sole validation mechanism - no separate held-out test set. At n=161, a
# holdout large enough for a precise standalone estimate is infeasible, and a
# smaller holdout gives a higher-variance estimate than repeated CV.
N_SPLITS  = 5
N_REPEATS = 10
RANDOM_STATE = 42

# Toggle: True = full missingness -> NZV -> p-value -> bootstrap stability
# selection (original method). False = stop after p-value filtering, skip
# bootstrap stability entirely.
USE_BOOTSTRAP = False

# Outer folds run in parallel (one process per fold, up to N_JOBS at once).
# Capped near N_SPLITS*N_REPEATS since that's the max useful parallelism for
# this loop - going higher wastes workers since there are only 50 folds.
N_JOBS       = 48
N_JOBS_INNER = 1

N_TOTAL_CORES = 112  # fold-parallel workers for the FS grid search

cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
# cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print(f"CV strategy: {N_SPLITS}-fold x {N_REPEATS} repeats = {N_SPLITS*N_REPEATS} total folds")
print(f"Models active: LR, RF"
      + (", SVM"      if USE_SVM      else "")
      + (", LightGBM" if USE_LIGHTGBM else "")
      + (", XGBoost"  if USE_XGBOOST  else ""))

## 3. Nested Feature Selection (Missingness + NZV + One-Hot + P-Value + Stability)

`FeatureSelector.fit()` runs on training-fold data only: missingness filter -> NZV filter -> one-hot encoding -> p-value filter (alpha=0.05) -> bootstrap stability selection -> `entering_features_`. Retained features are computed separately per model per fold after fitting (Section 4).

In [5]:
# ── One-hot encoder, fit per fold ──────────────────────────────────────────────
class FoldOneHotEncoder(BaseEstimator, TransformerMixin):
    """Fits on training-fold data: records observed categories per column,
    drops the FEATURE_VARS-specified reference category. transform() applies
    the same dummy columns to any data passed in; a category present in test
    but absent from training is encoded as all-zero."""

    def __init__(self, onehot_vars, onehot_ref):
        self.onehot_vars = onehot_vars
        self.onehot_ref  = onehot_ref

    def fit(self, X, y=None):
        self.categories_ = {}
        for col in self.onehot_vars:
            if col not in X.columns:
                continue
            observed = sorted(X[col].dropna().unique())
            ref = self.onehot_ref[col]
            self.categories_[col] = [c for c in observed if c != ref]
        return self

    def transform(self, X):
        X = X.copy()
        for col, cats in self.categories_.items():
            if col not in X.columns:
                continue
            for cat in cats:
                X[f'{col}_{int(cat)}'] = (X[col] == cat).astype(float)
            X = X.drop(columns=[col])
        return X

In [6]:
# ── Imputation strategy — CONTINUOUS columns get median, everything else
# (ORDINAL, BINARY, post-one-hot dummy columns) gets most_frequent.

def base_name_for_type(col):
    """Map a one-hot-expanded column name back to its FEATURE_VARS key."""
    if col in FEATURE_VARS:
        return col
    stripped = col.rsplit('_', 1)[0]
    if stripped in FEATURE_VARS:
        return stripped
    return col

class _OrderPreservingImputer(BaseEstimator, TransformerMixin):
    """Wraps ColumnTransformer and reindexes its output back to feature_cols,
    since ColumnTransformer groups output by transformer, not input order."""

    def __init__(self, feature_cols, cont_cols, other_cols):
        self.feature_cols = feature_cols
        self.cont_cols    = cont_cols
        self.other_cols   = other_cols

    def fit(self, X, y=None):
        transformers = []
        if self.cont_cols:
            transformers.append(('cont', SimpleImputer(strategy='median'), self.cont_cols))
        if self.other_cols:
            transformers.append(('other', SimpleImputer(strategy='most_frequent'), self.other_cols))
        self.ct_ = ColumnTransformer(transformers, remainder='drop')
        self.ct_.fit(X)
        return self

    def transform(self, X):
        out = self.ct_.transform(X)
        out_cols = self.cont_cols + self.other_cols  # ColumnTransformer's actual emission order
        df = pd.DataFrame(out, columns=out_cols, index=X.index)
        return df[self.feature_cols]

def make_imputer(feature_cols):
    cont_cols = [c for c in feature_cols
                 if base_name_for_type(c) in FEATURE_VARS
                 and get_var_type(base_name_for_type(c)) == 'CONTINUOUS']
    other_cols = [c for c in feature_cols if c not in cont_cols]
    return _OrderPreservingImputer(list(feature_cols), cont_cols, other_cols)

In [7]:
# ── FeatureSelector — produces the ENTERING feature set, fit per fold ─────────
class FeatureSelector(BaseEstimator, TransformerMixin):
    """fit(X_train, y_train) runs missingness -> NZV -> one-hot -> p-value ->
    bootstrap stability selection, storing the result as entering_features_.
    transform(X) applies the same one-hot encoding and subsets to that set.

    Bootstrap importance (_get_feature_ranking) is computed in-sample, i.e. on
    the same resample a model was fit on. This is intentional: stability
    selection measures how consistently a feature ranks highly across
    resamples, not unbiased predictive importance, so no held-out split is
    needed at this stage."""

    def __init__(self, miss_threshold=40, nzv_threshold=0.95, alpha=0.05,
                 n_bootstrap=100, top_k=25, stability_threshold=0.5,
                 consensus_frac=2/3, onehot_vars=None, onehot_ref=None,
                 random_state=42, use_bootstrap=True):
        self.miss_threshold       = miss_threshold
        self.nzv_threshold        = nzv_threshold
        self.alpha                = alpha
        self.n_bootstrap          = n_bootstrap
        self.top_k                = top_k
        self.stability_threshold  = stability_threshold
        self.consensus_frac       = consensus_frac
        self.onehot_vars          = onehot_vars or []
        self.onehot_ref           = onehot_ref or {}
        self.random_state         = random_state
        self.use_bootstrap        = use_bootstrap

    def _get_feature_ranking(self, model_name, pipe, X_fit, y_fit):
        pipe.fit(X_fit, y_fit)

        if model_name == 'Logistic Regression':
            lr_model = pipe['model']
            importances = pd.Series(np.abs(lr_model.coef_[0]), index=X_fit.columns)
        else:
            if model_name == 'Random Forest':
                model_obj  = pipe['model']
                X_shap_imp = pipe['imputer'].transform(X_fit)
                X_shap     = pd.DataFrame(X_shap_imp, columns=X_fit.columns)
            else:
                model_obj = pipe
                X_shap    = X_fit.copy()

            explainer   = shap.TreeExplainer(model_obj)
            shap_values = explainer.shap_values(X_shap)

            shap_vals = np.array(shap_values)
            if shap_vals.ndim == 3:
                shap_vals = shap_vals[:, :, 1]
            elif isinstance(shap_values, list):
                shap_vals = np.array(shap_values[1])
            else:
                shap_vals = shap_values

            importances = pd.Series(np.abs(shap_vals).mean(axis=0), index=X_fit.columns)

        return importances.sort_values(ascending=False)

    def fit(self, X, y):
        X = pd.DataFrame(X).copy()
        y = np.asarray(y)
        cols = list(X.columns)

        # Missingness filter
        miss_pct = X.isnull().sum() / len(X) * 100
        cols = [c for c in cols if miss_pct[c] < self.miss_threshold]

        # NZV filter
        kept = []
        for c in cols:
            series = X[c].dropna()
            if len(series) == 0:
                continue
            dom_pct = series.value_counts(normalize=True).iloc[0]
            if dom_pct < self.nzv_threshold:
                kept.append(c)
        cols = kept

        # One-hot encode multi-category vars
        onehot_in_cols = [c for c in cols if c in self.onehot_vars]
        self.encoder_ = FoldOneHotEncoder(onehot_in_cols, self.onehot_ref)
        self.encoder_.fit(X[cols])
        X_enc = self.encoder_.transform(X[cols])
        cols  = list(X_enc.columns)

        # P-value filter (alpha=0.05).
        # CONTINUOUS vs target: Shapiro-Wilk checks normality of each group
        # separately; Welch's t-test if both groups pass (p>0.05), otherwise
        # Mann-Whitney U (robust to non-normal/small-n distributions).
        # ORDINAL/BINARY/ONEHOT-dummy vs target: contingency table test.
        # Cochran's rule on expected cell counts decides Fisher's exact
        # (2x2 tables with any expected count <5) vs chi-square (otherwise).
        # Larger-than-2x2 tables failing the 20%-of-cells-<5 rule have no
        # exact alternative in scipy without simulation, so chi-square is
        # used with a recorded caveat rather than an invalid substitute.
        p_values     = {}
        test_methods = {}
        for c in cols:
            series = X_enc[c]
            base   = base_name_for_type(c)
            is_cont = base in FEATURE_VARS and get_var_type(base) == 'CONTINUOUS'

            paired = pd.DataFrame({'x': series, 'y': y}).dropna()
            if len(paired) < 10 or paired['x'].nunique() < 2:
                p_values[c] = 1.0
                test_methods[c] = 'skipped_insufficient_data'
                continue

            try:
                if is_cont:
                    g0 = paired.loc[paired['y'] == 0, 'x']
                    g1 = paired.loc[paired['y'] == 1, 'x']
                    if len(g0) < 3 or len(g1) < 3:
                        _, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
                        p_values[c] = p
                        test_methods[c] = 'mannwhitneyu_insufficient_n_for_shapiro'
                    else:
                        _, p_norm0 = stats.shapiro(g0)
                        _, p_norm1 = stats.shapiro(g1)
                        if p_norm0 > 0.05 and p_norm1 > 0.05:
                            _, p = stats.ttest_ind(g0, g1, equal_var=False)
                            p_values[c] = p
                            test_methods[c] = 'welch_ttest'
                        else:
                            _, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
                            p_values[c] = p
                            test_methods[c] = 'mannwhitneyu'
                else:
                    contingency = pd.crosstab(paired['x'], paired['y'])
                    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
                        p_values[c] = 1.0
                        test_methods[c] = 'skipped_degenerate_table'
                    else:
                        expected = stats.contingency.expected_freq(contingency.values)
                        pct_low  = (expected < 5).mean()
                        is_2x2   = contingency.shape == (2, 2)

                        if is_2x2 and (expected < 5).any():
                            _, p = stats.fisher_exact(contingency.values)
                            p_values[c] = p
                            test_methods[c] = 'fisher_exact'
                        else:
                            _, p, _, _ = stats.chi2_contingency(contingency)
                            p_values[c] = p
                            test_methods[c] = ('chi2_low_expected_count_caveat'
                                                if pct_low > 0.20 else 'chi2')
            except Exception:
                p_values[c] = 1.0
                test_methods[c] = 'skipped_exception'

        self.p_values_    = pd.Series(p_values).sort_values()
        self.test_methods_ = pd.Series(test_methods)
        cols = [c for c in cols if p_values[c] < self.alpha]
        self.n_after_pvalue_ = len(cols)

        if not self.use_bootstrap:
            # Skip bootstrap stability selection entirely - entering set is
            # whatever survives missingness -> NZV -> p-value alone. Far more
            # stable fold to fold, but loses the resample-robustness check
            # stability selection provides (guards against a variable passing
            # p-value filtering by chance on one particular training sample).
            self._bootstrap_rankings_ = None
            self._stab_columns_       = None
            self.entering_features_   = list(cols)
            return self

        # Bootstrap stability selection — cross-model consensus across resamples
        X_stab = X_enc[cols].astype(float)

        base_models = {
            'Logistic Regression': Pipeline([
                ('imputer', make_imputer(cols)),
                ('scaler',  StandardScaler()),
                ('model',   LogisticRegression(
                    penalty='elasticnet', solver='saga', C=1.0, l1_ratio=0.5,
                    class_weight='balanced', max_iter=5000, random_state=self.random_state,
                ))
            ]),
            'Random Forest': Pipeline([
                ('imputer', make_imputer(cols)),
                ('model',   RandomForestClassifier(
                    n_estimators=500, min_samples_leaf=5, class_weight='balanced',
                    random_state=self.random_state, n_jobs=N_JOBS_INNER,
                ))
            ]),
            'XGBoost': XGBClassifier(
                objective='binary:logistic', max_depth=4, learning_rate=0.05,
                n_estimators=500, min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
                scale_pos_weight=(y == 0).sum() / max((y == 1).sum(), 1),
                tree_method='hist', random_state=self.random_state, n_jobs=N_JOBS_INNER, verbosity=0,
            ),
        }

        self._bootstrap_rankings_, self._stab_columns_ = self._run_bootstraps(X_stab, y, base_models)
        self.entering_features_ = self._mask_from_rankings(
            self._bootstrap_rankings_, self._stab_columns_,
            self.top_k, self.stability_threshold, self.consensus_frac)
        return self

    def _run_bootstraps(self, X_stab, y, base_models):
        """n_bootstrap model fits per base model, stores the full ranking per
        (model, bootstrap). top_k is applied later in _mask_from_rankings."""
        rankings = {name: [] for name in base_models}
        for model_name, pipe in base_models.items():
            for b in range(self.n_bootstrap):
                X_boot, y_boot = resample(
                    X_stab, y, replace=True, n_samples=len(X_stab),
                    random_state=self.random_state + b, stratify=y
                )
                rankings[model_name].append(self._get_feature_ranking(model_name, pipe, X_boot, y_boot))
        return rankings, list(X_stab.columns)

    @staticmethod
    def _mask_from_rankings(rankings, columns, top_k, stability_threshold, consensus_frac):
        """Re-slices stored rankings for any top_k/stability_threshold/consensus_frac."""
        stability_results = {}
        for model_name, model_rankings in rankings.items():
            selection_counts = pd.Series(0, index=columns)
            for ranking in model_rankings:
                selection_counts.loc[ranking.head(top_k).index] += 1
            stability_results[model_name] = selection_counts / len(model_rankings)
        stability_df = pd.DataFrame(stability_results)
        min_models_needed = np.ceil(consensus_frac * stability_df.shape[1])
        mask = (stability_df >= stability_threshold).sum(axis=1) >= min_models_needed
        return stability_df[mask].index.tolist()

    def transform(self, X):
        X_enc = self.encoder_.transform(X[[c for c in X.columns if c in self.entering_features_
                                            or c in self.encoder_.onehot_vars]])
        return X_enc[self.entering_features_].astype(float)

In [8]:
# ── Retained features — computed per model, per fold, after fitting ───────────
def compute_retained_features(pipe, model_name, entering_cols):
    """Non-zero coefficient (LR) or feature_importances_ (RF/XGBoost/LightGBM).
    SVM exposes no per-feature weight, so all entering features are retained."""
    model = pipe['model'] if isinstance(pipe, Pipeline) else pipe

    if model_name == 'Logistic Regression':
        weights = np.abs(model.coef_[0])
    elif model_name == 'SVM':
        return list(entering_cols)
    else:
        weights = model.feature_importances_

    return [c for c, w in zip(entering_cols, weights) if w > 0]

## 4. Primary-Only Pipeline (Entering -> Fit -> Retained -> Nested CV)

Feature selection runs once across all 50 outer folds, caching the fitted `FeatureSelector` objects for reuse across base CV, tuned CV, and ensemble.

In [9]:
# ── Grid search: top_k / stability_threshold / consensus_frac / alpha ─────────
# consensus_frac = fraction of voting models that must clear stability_threshold.
RUN_FS_GRID_SEARCH = True

FS_GRID = {
    'top_k': [15, 20, 25, 30, 35],
    'stability_threshold': [0.4, 0.5, 0.6],
    'consensus_frac': [1/3, 2/3, 1.0],
    'alpha': [0.01, 0.05, 0.1, 0.15],
}

def _fit_fold_for_grid(train_idx, X_pool, y, onehot_vars, onehot_ref, base_config):
    """P-value computation only, shared across all grid combinations."""
    X_train, y_train = X_pool.iloc[train_idx], y.iloc[train_idx]
    selector = FeatureSelector(
        miss_threshold=base_config['miss_threshold'], nzv_threshold=base_config['nzv_threshold'],
        n_bootstrap=base_config['n_bootstrap'],
        random_state=base_config['random_state'], onehot_vars=onehot_vars, onehot_ref=onehot_ref,
    )
    X = pd.DataFrame(X_train).copy()
    y_arr = np.asarray(y_train)
    cols = list(X.columns)
    miss_pct = X.isnull().sum() / len(X) * 100
    cols = [c for c in cols if miss_pct[c] < selector.miss_threshold]
    kept = []
    for c in cols:
        series = X[c].dropna()
        if len(series) == 0:
            continue
        if series.value_counts(normalize=True).iloc[0] < selector.nzv_threshold:
            kept.append(c)
    cols = kept
    onehot_in_cols = [c for c in cols if c in selector.onehot_vars]
    selector.encoder_ = FoldOneHotEncoder(onehot_in_cols, selector.onehot_ref)
    selector.encoder_.fit(X[cols])
    X_enc = selector.encoder_.transform(X[cols])
    cols = list(X_enc.columns)

    p_values = {}
    for c in cols:
        series = X_enc[c]
        base = base_name_for_type(c)
        is_cont = base in FEATURE_VARS and get_var_type(base) == 'CONTINUOUS'
        paired = pd.DataFrame({'x': series, 'y': y_arr}).dropna()
        if len(paired) < 10 or paired['x'].nunique() < 2:
            p_values[c] = 1.0
            continue
        try:
            if is_cont:
                g0 = paired.loc[paired['y'] == 0, 'x']
                g1 = paired.loc[paired['y'] == 1, 'x']
                if len(g0) < 3 or len(g1) < 3:
                    _, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
                else:
                    _, p_norm0 = stats.shapiro(g0)
                    _, p_norm1 = stats.shapiro(g1)
                    if p_norm0 > 0.05 and p_norm1 > 0.05:
                        _, p = stats.ttest_ind(g0, g1, equal_var=False)
                    else:
                        _, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
                p_values[c] = p
            else:
                contingency = pd.crosstab(paired['x'], paired['y'])
                if contingency.shape[0] < 2 or contingency.shape[1] < 2:
                    p_values[c] = 1.0
                else:
                    expected = stats.contingency.expected_freq(contingency.values)
                    is_2x2 = contingency.shape == (2, 2)
                    if is_2x2 and (expected < 5).any():
                        _, p = stats.fisher_exact(contingency.values)
                    else:
                        _, p, _, _ = stats.chi2_contingency(contingency)
                    p_values[c] = p
        except Exception:
            p_values[c] = 1.0
    return p_values, cols, X_enc, y_train, selector

def _bootstrap_for_alpha(alpha, p_values, cols, X_enc, y_train, selector):
    cols = [c for c in cols if p_values[c] < alpha]
    if not cols:
        return {}, [], selector.encoder_, cols
    X_stab = X_enc[cols].astype(float)
    y_arr = np.asarray(y_train)
    base_models = {
        'Logistic Regression': Pipeline([
            ('imputer', make_imputer(cols)), ('scaler', StandardScaler()),
            ('model', LogisticRegression(penalty='elasticnet', solver='saga', C=1.0, l1_ratio=0.5,
                                          class_weight='balanced', max_iter=5000,
                                          random_state=selector.random_state))
        ]),
        'Random Forest': Pipeline([
            ('imputer', make_imputer(cols)),
            ('model', RandomForestClassifier(n_estimators=500, min_samples_leaf=5,
                                              class_weight='balanced',
                                              random_state=selector.random_state, n_jobs=N_JOBS_INNER))
        ]),
        'XGBoost': XGBClassifier(
            objective='binary:logistic', max_depth=4, learning_rate=0.05, n_estimators=500,
            min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=(y_arr == 0).sum() / max((y_arr == 1).sum(), 1),
            tree_method='hist', random_state=selector.random_state, n_jobs=N_JOBS_INNER, verbosity=0,
        ),
    }
    rankings, stab_columns = selector._run_bootstraps(X_stab, y_train, base_models)
    return rankings, stab_columns, selector.encoder_, cols

def _score_combo_given_rankings(tk, st, cm, rankings, stab_columns, encoder, X_train, X_test,
                                 y_train, y_test, base_config):
    """Cheap: masks the already-computed rankings for this combo, scores with a quick LR."""
    entering = FeatureSelector._mask_from_rankings(rankings, stab_columns, tk, st, cm)
    if not entering:
        return 0.5
    def _transform(X):
        X_enc = encoder.transform(X[[c for c in X.columns if c in entering or c in encoder.onehot_vars]])
        return X_enc[entering].astype(float)
    X_tr_sel, X_te_sel = _transform(X_train), _transform(X_test)
    clf = Pipeline([('imputer', make_imputer(entering)), ('scaler', StandardScaler()),
                     ('model', LogisticRegression(max_iter=2000, class_weight='balanced',
                                                   random_state=base_config['random_state']))])
    clf.fit(X_tr_sel, y_train)
    return roc_auc_score(y_test, clf.predict_proba(X_te_sel)[:, 1])

def _score_one_fold_all_combos(train_idx, test_idx, X_pool, y, onehot_vars, onehot_ref, base_config, combos):
    """Bootstraps once per (fold, alpha), scores top_k/stability/consensus per bootstrap."""
    p_values, cols, X_enc, y_train, selector = _fit_fold_for_grid(
        train_idx, X_pool, y, onehot_vars, onehot_ref, base_config)
    X_train, X_test = X_pool.iloc[train_idx], X_pool.iloc[test_idx]
    y_test = y.iloc[test_idx]

    alphas = sorted(set(a for _, _, _, a in combos))
    aucs_by_combo = {}
    n_after_pvalue_by_alpha = {}
    for alpha in alphas:
        rankings, stab_columns, encoder, alpha_cols = _bootstrap_for_alpha(
            alpha, p_values, cols, X_enc, y_train, selector)
        n_after_pvalue_by_alpha[alpha] = len(alpha_cols)
        for tk, st, cm, a in combos:
            if a != alpha:
                continue
            aucs_by_combo[(tk, st, cm, a)] = _score_combo_given_rankings(
                tk, st, cm, rankings, stab_columns, encoder, X_train, X_test, y_train, y_test, base_config)

    aucs = [aucs_by_combo[combo] for combo in combos]
    return aucs, n_after_pvalue_by_alpha

def run_fs_grid_search(X_pool, y, fold_splits, onehot_vars, onehot_ref, base_config, label):
    combos = [(tk, st, cm, a) for tk in FS_GRID['top_k']
                              for st in FS_GRID['stability_threshold']
                              for cm in FS_GRID['consensus_frac']
                              for a in FS_GRID['alpha']]
    print(f"Grid search ({label}): {len(combos)} combinations, {len(fold_splits)} folds, {N_TOTAL_CORES} workers")

    _t0 = time.time()
    fold_results = Parallel(n_jobs=N_TOTAL_CORES)(
        delayed(_score_one_fold_all_combos)(train_idx, test_idx, X_pool, y,
                                             onehot_vars, onehot_ref, base_config, combos)
        for train_idx, test_idx in tqdm(fold_splits, desc=f'{label} fs grid (folds)')
    )

    per_fold_combo_aucs = [r[0] for r in fold_results]
    print(f"Bootstrap+scoring across all folds: {time.time()-_t0:.1f}s")
    print("Columns surviving p-value filter, per alpha:")
    for alpha in FS_GRID['alpha']:
        counts = [r[1][alpha] for r in fold_results]
        print(f"  alpha={alpha}: mean={np.mean(counts):.1f}, min={min(counts)}, max={max(counts)}")
    mean_aucs = np.round(np.mean(np.array(per_fold_combo_aucs), axis=0), 6)

    grid_df = pd.DataFrame(
        [(tk, st, cm, a, auc) for (tk, st, cm, a), auc in zip(combos, mean_aucs)],
        columns=['top_k', 'stability_threshold', 'consensus_frac', 'alpha', 'mean_auc']
    )
    grid_df = grid_df.sort_values(
        ['mean_auc', 'stability_threshold', 'consensus_frac', 'top_k', 'alpha'],
        ascending=[False, False, False, True, True]
    ).reset_index(drop=True)

    print(f"\nTop 5 combinations ({label}):")
    print(grid_df.head(5).to_string(index=False))
    grid_df.to_csv(f'{OUTPUT_DIR}/fs_grid_search_{label}.csv', index=False)
    print(f"Saved: fs_grid_search_{label}.csv")
    return grid_df


In [10]:
X_primary = X_all[PRIMARY_VARS].copy()

if RUN_FS_GRID_SEARCH:
    # alpha here is an inert placeholder - FeatureSelector.__init__ requires it,
    # but _fit_fold_for_grid never applies it; the real sweep happens per
    # combination inside _bootstrap_for_alpha via FS_GRID['alpha'].
    _fs_base_config = {'miss_threshold': 40, 'nzv_threshold': 0.95, 'alpha': 0.05,
                        'n_bootstrap': 100, 'random_state': RANDOM_STATE}
    primary_fs_grid = run_fs_grid_search(
        X_primary, y, list(cv.split(X_primary, y)), ONEHOT_VARS, ONEHOT_REF,
        _fs_base_config, 'primary'
    )

def _fit_one_selector(train_idx, X_pool, y, config, onehot_vars, onehot_ref):
    """Fits one fold's FeatureSelector. Runs in its own process under Parallel."""
    X_train, y_train = X_pool.iloc[train_idx], y.iloc[train_idx]
    selector = FeatureSelector(
        miss_threshold=config['miss_threshold'], nzv_threshold=config['nzv_threshold'],
        alpha=config['alpha'], n_bootstrap=config['n_bootstrap'], top_k=config['top_k'],
        stability_threshold=config['stability_threshold'],
        consensus_frac=config['consensus_frac'],
        random_state=config['random_state'], onehot_vars=onehot_vars, onehot_ref=onehot_ref,
        use_bootstrap=config.get('use_bootstrap', True),
    )
    selector.fit(X_train, y_train)
    return selector

def median_params(param_list, int_keys=()):
    """Median across folds for numeric params; mode (most common value) for
    categorical params (e.g. tune_rf's max_features, which ranges over
    'sqrt'/'log2'/None — np.median can't sort a mix of str and NoneType)."""
    if not param_list:
        return {}
    keys = param_list[0].keys()
    out = {}
    for k in keys:
        vals = [p[k] for p in param_list]
        if all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in vals):
            med = float(np.median(vals))
            out[k] = int(round(med)) if k in int_keys else med
        else:
            out[k] = Counter(vals).most_common(1)[0][0]
    return out

PRIMARY_CACHE_PATH = f'{OUTPUT_DIR}/feature_selection_cache_primary.pkl'
PRIMARY_CONFIG = {
    'miss_threshold': 40, 'nzv_threshold': 0.95, 'alpha': 0.05, 'n_bootstrap': 100,
    'top_k': 25, 'stability_threshold': 0.5, 'consensus_frac': 2/3,
    'random_state': RANDOM_STATE, 'use_bootstrap': USE_BOOTSTRAP,
    'n_splits': N_SPLITS, 'n_repeats': N_REPEATS,
    'var_set': 'primary', 'data_hash': DATA_HASH,
}

if RUN_FS_GRID_SEARCH:
    _winner = primary_fs_grid.iloc[0]
    PRIMARY_CONFIG['top_k']               = int(_winner['top_k'])
    PRIMARY_CONFIG['stability_threshold'] = float(_winner['stability_threshold'])
    PRIMARY_CONFIG['consensus_frac']      = float(_winner['consensus_frac'])
    PRIMARY_CONFIG['alpha']               = float(_winner['alpha'])
    print(f"Applied grid winner to PRIMARY_CONFIG: top_k={PRIMARY_CONFIG['top_k']}, "
          f"stability_threshold={PRIMARY_CONFIG['stability_threshold']}, "
          f"consensus_frac={PRIMARY_CONFIG['consensus_frac']:.3f}, alpha={PRIMARY_CONFIG['alpha']}")

def run_or_load_selection(X_pool, cache_path, config, onehot_vars, onehot_ref):
    """Returns (fold_splits, fold_selectors), one selector per outer fold."""
    if os.path.exists(cache_path):
        with open(cache_path, 'rb') as f:
            cache = pickle.load(f)
        if cache['config'] == config:
            print(f"Loaded cached feature selection from: {cache_path}")
            print(f"  {len(cache['fold_splits'])} folds, config matches - skipping recomputation")
            return cache['fold_splits'], cache['fold_selectors']

    fold_splits = list(cv.split(X_pool, y))
    n_total = len(fold_splits)
    print(f"Running feature selection: {n_total} folds on {N_JOBS} workers")

    fold_selectors = list(tqdm(
        Parallel(n_jobs=N_JOBS, return_as='generator')(
            delayed(_fit_one_selector)(train_idx, X_pool, y, config, onehot_vars, onehot_ref)
            for train_idx, test_idx in fold_splits
        ),
        total=n_total, desc='feature selection', mininterval=5.0
    ))

    for fold_idx, selector in enumerate(fold_selectors):
        print(f"  fold {fold_idx+1}/{n_total} | entering={len(selector.entering_features_)}")

    with open(cache_path, 'wb') as f:
        pickle.dump({'fold_splits': fold_splits, 'fold_selectors': fold_selectors,
                     'config': config}, f)
    print(f"Cached to: {cache_path}")
    return fold_splits, fold_selectors

print("Running primary-only feature selection...")
primary_fold_splits, primary_fold_selectors = run_or_load_selection(
    X_primary, PRIMARY_CACHE_PATH, PRIMARY_CONFIG, ONEHOT_VARS, ONEHOT_REF
)

primary_fold_entering = [s.entering_features_ for s in primary_fold_selectors]
mean_entering = np.mean([len(e) for e in primary_fold_entering])
print(f"\nPrimary-only selection ready: {len(primary_fold_splits)} folds")
print(f"  Mean entering features/fold: {mean_entering:.1f} / {len(PRIMARY_VARS)} candidates")

In [11]:
# ── Base pipelines — every entry is a builder function, never a pre-built
# instance, so run_base_cv / run_ensemble_cv get a fresh, unfit model per fold.

def build_base_lr(feature_cols, y_tr=None):
    return Pipeline([
        ('imputer', make_imputer(feature_cols)),
        ('scaler',  StandardScaler()),
        ('model',   LogisticRegression(
            penalty='elasticnet', solver='saga', l1_ratio=0.5, C=1.0,
            class_weight='balanced', max_iter=5000, random_state=RANDOM_STATE,
        ))
    ])

def build_base_rf(feature_cols, y_tr=None):
    return Pipeline([
        ('imputer', make_imputer(feature_cols)),
        ('model',   RandomForestClassifier(
            n_estimators=500, max_depth=None, min_samples_leaf=5,
            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS_INNER,
        ))
    ])

def build_base_svm(feature_cols, y_tr=None):
    return Pipeline([
        ('imputer', make_imputer(feature_cols)),
        ('scaler',  StandardScaler()),
        ('model',   CalibratedClassifierCV(
            SVC(kernel='rbf', C=1.0, class_weight='balanced', random_state=RANDOM_STATE),
            cv=3,
        ))
    ])

def build_base_lgb(feature_cols, y_tr=None):
    return LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=4, num_leaves=15,
        min_child_samples=10, is_unbalance=True, colsample_bytree=0.8, subsample=0.8,
        subsample_freq=1, reg_alpha=0.1, reg_lambda=1.0,
        random_state=RANDOM_STATE, n_jobs=N_JOBS_INNER, verbose=-1,
    )

def build_base_xgb(feature_cols, y_tr=None):
    # Falls back to global y only if called without fold-local labels.
    y_ref = y_tr if y_tr is not None else y
    return XGBClassifier(
        objective='binary:logistic', n_estimators=500, learning_rate=0.05, max_depth=4,
        min_child_weight=5, scale_pos_weight=(y_ref==0).sum() / (y_ref==1).sum(),
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        tree_method='hist', random_state=RANDOM_STATE, n_jobs=N_JOBS_INNER, verbosity=0,
    )

BASE_MODELS = {'Logistic Regression': build_base_lr, 'Random Forest': build_base_rf}
if USE_SVM:      BASE_MODELS['SVM']      = build_base_svm
if USE_LIGHTGBM: BASE_MODELS['LightGBM'] = build_base_lgb
if USE_XGBOOST:  BASE_MODELS['XGBoost']  = build_base_xgb

print("Base pipelines defined:")
for name in BASE_MODELS:
    print(f"  {name}")

In [12]:
# ── Base CV — reuses the cached selector per fold, tracks entering + retained ──
def compute_metrics(y_true, y_pred_proba, y_pred_class):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_class).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        'auc':         roc_auc_score(y_true, y_pred_proba),
        'f1':          f1_score(y_true, y_pred_class, zero_division=0),
        'precision':   precision_score(y_true, y_pred_class, zero_division=0),
        'brier':       brier_score_loss(y_true, y_pred_proba),
        'bal_acc':     balanced_accuracy_score(y_true, y_pred_class),
        'sensitivity': sensitivity,
        'specificity': specificity,
        'mcc':         matthews_corrcoef(y_true, y_pred_class),
    }

def _base_cv_one_fold(name, build_fn, train_idx, test_idx, selector, X_pool, y):
    """One outer fold's base-CV work for one model. Runs in its own process
    under Parallel - everything it touches (X_pool, y, selector) is read-only."""
    X_train, X_test = X_pool.iloc[train_idx], X_pool.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    X_train_sel = selector.transform(X_train)
    X_test_sel  = selector.transform(X_test)
    entering_cols = selector.entering_features_

    pipe = build_fn(entering_cols, y_train)
    pipe.fit(X_train_sel, y_train)

    retained = compute_retained_features(pipe, name, entering_cols)

    y_proba = pipe.predict_proba(X_test_sel)[:, 1]
    y_pred  = pipe.predict(X_test_sel)
    y_train_proba = pipe.predict_proba(X_train_sel)[:, 1]
    train_auc = roc_auc_score(y_train, y_train_proba)

    metrics = compute_metrics(y_test, y_proba, y_pred)
    metrics['train_auc'] = train_auc
    return metrics, len(retained)

def run_base_cv(X_pool, fold_splits, fold_selectors, models, label):
    results    = {name: [] for name in models}
    n_retained = {name: [] for name in models}

    for name, build_fn in models.items():
        print(f"Running base CV ({label}): {name} - {len(fold_splits)} folds on {N_JOBS} workers")

        fold_pairs = list(zip(fold_splits, fold_selectors))
        fold_outputs = list(tqdm(
            Parallel(n_jobs=N_JOBS, return_as='generator')(
                delayed(_base_cv_one_fold)(name, build_fn, train_idx, test_idx, selector, X_pool, y)
                for (train_idx, test_idx), selector in fold_pairs
            ),
            total=len(fold_pairs), desc=f"{label}/{name}", mininterval=5.0
        ))

        fold_metrics = [out[0] for out in fold_outputs]
        results[name]    = fold_metrics
        n_retained[name] = [out[1] for out in fold_outputs]

        mean_auc = np.mean([m['auc'] for m in fold_metrics])
        mean_ret = np.mean(n_retained[name])
        print(f"  done | mean AUC = {mean_auc:.3f} | mean retained = {mean_ret:.1f}")

    n_entering = [len(s.entering_features_) for s in fold_selectors]
    return results, n_entering, n_retained

primary_base_results, primary_n_entering, primary_n_retained_base = run_base_cv(
    X_primary, primary_fold_splits, primary_fold_selectors, BASE_MODELS, 'primary'
)
print(f"\nPrimary base CV complete: {N_SPLITS*N_REPEATS} folds")

## 5. Hyperparameter Tuning & Tuned CV - Primary Only

Tuning re-runs fresh inside each outer fold on that fold's entering features,
using an inner 5-fold stratified CV on the training portion only.

In [13]:
N_TRIALS    = 100   # set 0 to skip tuning
TUNE_METRIC = 'roc_auc'

# Fold-level pruning: reports min(mean, median) of inner-CV scores after each
# fold. n_warmup_steps < N_SPLITS, n_startup_trials=10 unpruned trials first.
PRUNER = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2)

def _scored_fit(model, X_tr, y_tr, inner_cv, trial):
    """Fits one fold at a time, reporting min(mean, median) of completed AUCs
    to Optuna after each fold and raising TrialPruned if Optuna says to stop."""
    scores = []
    for step, (tr_idx, val_idx) in enumerate(inner_cv.split(X_tr, y_tr)):
        X_fold_tr, X_fold_val = X_tr.iloc[tr_idx], X_tr.iloc[val_idx]
        y_fold_tr, y_fold_val = y_tr.iloc[tr_idx], y_tr.iloc[val_idx]
        model.fit(X_fold_tr, y_fold_tr)
        proba = model.predict_proba(X_fold_val)[:, 1]
        scores.append(roc_auc_score(y_fold_val, proba))

        reported = min(np.mean(scores), np.median(scores))
        trial.report(reported, step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(scores)

def tune_lr(X_tr, y_tr, n_trials):
    inner_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    feature_cols = list(X_tr.columns)
    def objective(trial):
        scaler_choice = trial.suggest_categorical('scaler', ['standard', 'robust'])
        scaler = StandardScaler() if scaler_choice == 'standard' else RobustScaler()
        cw_choice = trial.suggest_categorical('class_weight', ['balanced', 'none'])
        model = Pipeline([
            ('imputer', make_imputer(feature_cols)),
            ('scaler',  scaler),
            ('model',   LogisticRegression(
                penalty='elasticnet', solver='saga',
                C=trial.suggest_float('C', 1e-3, 10.0, log=True),
                l1_ratio=trial.suggest_float('l1_ratio', 0.0, 1.0),
                class_weight=None if cw_choice == 'none' else 'balanced',
                max_iter=5000, random_state=RANDOM_STATE,
            ))
        ])
        return _scored_fit(model, X_tr, y_tr, inner_cv, trial)
    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                 pruner=PRUNER)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params

def tune_rf(X_tr, y_tr, n_trials):
    inner_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    feature_cols = list(X_tr.columns)
    def objective(trial):
        model = Pipeline([
            ('imputer', make_imputer(feature_cols)),
            ('model',   RandomForestClassifier(
                n_estimators=trial.suggest_int('n_estimators', 100, 1000, step=100),
                max_depth=trial.suggest_int('max_depth', 2, 20),
                min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 20),
                min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
                max_samples=trial.suggest_float('max_samples', 0.5, 1.0),
                max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
                class_weight='balanced',
                random_state=RANDOM_STATE, n_jobs=N_JOBS_INNER,
            ))
        ])
        return _scored_fit(model, X_tr, y_tr, inner_cv, trial)
    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                 pruner=PRUNER)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params

def tune_xgb(X_tr, y_tr, n_trials):
    inner_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    def objective(trial):
        model = XGBClassifier(
            objective='binary:logistic',
            max_depth=trial.suggest_int('max_depth', 2, 8),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            n_estimators=trial.suggest_int('n_estimators', 100, 1000, step=100),
            min_child_weight=trial.suggest_int('min_child_weight', 1, 10),
            gamma=trial.suggest_float('gamma', 1e-3, 5.0, log=True),
            subsample=trial.suggest_float('subsample', 0.5, 1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
            reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            scale_pos_weight=(y_tr==0).sum()/(y_tr==1).sum(),
            tree_method='hist',
            random_state=RANDOM_STATE, n_jobs=N_JOBS_INNER, verbosity=0,
        )
        return _scored_fit(model, X_tr, y_tr, inner_cv, trial)
    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                 pruner=PRUNER)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params

def build_tuned_lr(params, feature_cols):
    scaler = StandardScaler() if params.get('scaler', 'standard') == 'standard' else RobustScaler()
    cw = None if params.get('class_weight', 'balanced') == 'none' else 'balanced'
    return Pipeline([
        ('imputer', make_imputer(feature_cols)),
        ('scaler',  scaler),
        ('model',   LogisticRegression(
            penalty='elasticnet', solver='saga',
            C=params.get('C', 1.0), l1_ratio=params.get('l1_ratio', 0.5),
            class_weight=cw, max_iter=5000, random_state=RANDOM_STATE,
        ))
    ])

def build_tuned_rf(params, feature_cols):
    return Pipeline([
        ('imputer', make_imputer(feature_cols)),
        ('model',   RandomForestClassifier(
            n_estimators=params.get('n_estimators', 500),
            max_depth=params.get('max_depth', None),
            min_samples_leaf=params.get('min_samples_leaf', 5),
            min_samples_split=params.get('min_samples_split', 2),
            max_samples=params.get('max_samples', None),
            max_features=params.get('max_features', 'sqrt'),
            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS_INNER,
        ))
    ])

def build_tuned_xgb(params, y_tr):
    return XGBClassifier(
        objective='binary:logistic',
        max_depth=params.get('max_depth', 4),
        learning_rate=params.get('learning_rate', 0.05),
        n_estimators=params.get('n_estimators', 500),
        min_child_weight=params.get('min_child_weight', 5),
        gamma=params.get('gamma', 0.0),
        subsample=params.get('subsample', 0.8),
        colsample_bytree=params.get('colsample_bytree', 0.8),
        reg_alpha=params.get('reg_alpha', 0.1),
        reg_lambda=params.get('reg_lambda', 1.0),
        scale_pos_weight=(y_tr==0).sum()/(y_tr==1).sum(),
        tree_method='hist', random_state=RANDOM_STATE, n_jobs=N_JOBS_INNER, verbosity=0,
    )

def tune_svm(X_tr, y_tr, n_trials):
    inner_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    feature_cols = list(X_tr.columns)
    def objective(trial):
        kernel = trial.suggest_categorical('kernel', ['rbf', 'linear'])
        gamma  = trial.suggest_categorical('gamma', ['scale', 'auto']) if kernel == 'rbf' else 'scale'
        model = Pipeline([
            ('imputer', make_imputer(feature_cols)),
            ('scaler',  StandardScaler()),
            ('model',   CalibratedClassifierCV(
                SVC(
                    kernel=kernel,
                    C=trial.suggest_float('C', 1e-2, 100.0, log=True),
                    gamma=gamma,
                    class_weight='balanced', random_state=RANDOM_STATE,
                ), cv=3,
            ))
        ])
        return _scored_fit(model, X_tr, y_tr, inner_cv, trial)
    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                 pruner=PRUNER)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params

def tune_lgb(X_tr, y_tr, n_trials):
    inner_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    def objective(trial):
        model = LGBMClassifier(
            num_leaves=trial.suggest_int('num_leaves', 8, 64),
            max_depth=trial.suggest_int('max_depth', 3, 8),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            n_estimators=trial.suggest_int('n_estimators', 100, 1000, step=100),
            min_child_samples=trial.suggest_int('min_child_samples', 5, 30),
            min_split_gain=trial.suggest_float('min_split_gain', 1e-3, 5.0, log=True),
            subsample=trial.suggest_float('subsample', 0.5, 1.0),
            subsample_freq=1,
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
            reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            is_unbalance=True,
            random_state=RANDOM_STATE, n_jobs=N_JOBS_INNER, verbose=-1,
        )
        return _scored_fit(model, X_tr, y_tr, inner_cv, trial)
    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                 pruner=PRUNER)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params

def build_tuned_svm(params, feature_cols):
    return Pipeline([
        ('imputer', make_imputer(feature_cols)),
        ('scaler',  StandardScaler()),
        ('model',   CalibratedClassifierCV(
            SVC(
                kernel=params.get('kernel', 'rbf'),
                C=params.get('C', 1.0),
                gamma=params.get('gamma', 'scale'),
                class_weight='balanced', random_state=RANDOM_STATE,
            ), cv=3,
        ))
    ])

def build_tuned_lgb(params):
    return LGBMClassifier(
        num_leaves=params.get('num_leaves', 15),
        max_depth=params.get('max_depth', 4),
        learning_rate=params.get('learning_rate', 0.05),
        n_estimators=params.get('n_estimators', 500),
        min_child_samples=params.get('min_child_samples', 10),
        min_split_gain=params.get('min_split_gain', 0.0),
        subsample=params.get('subsample', 0.8),
        subsample_freq=1,
        colsample_bytree=params.get('colsample_bytree', 0.8),
        reg_alpha=params.get('reg_alpha', 0.1),
        reg_lambda=params.get('reg_lambda', 1.0),
        is_unbalance=True,
        random_state=RANDOM_STATE, n_jobs=N_JOBS_INNER, verbose=-1,
    )

# Fingerprints these functions' source so the tuning cache auto-invalidates
# on any search-space change.
_TUNE_SEARCH_SPACE_FNS = [
    _scored_fit, tune_lr, tune_rf, tune_xgb, tune_svm, tune_lgb,
    build_tuned_lr, build_tuned_rf, build_tuned_xgb, build_tuned_svm, build_tuned_lgb,
]
TUNE_SEARCH_SPACE_HASH = hashlib.md5(
    ''.join(inspect.getsource(fn) for fn in _TUNE_SEARCH_SPACE_FNS).encode()
).hexdigest()

print(f"Tuning configured: N_TRIALS={N_TRIALS} per outer fold per model")
print(f"Search-space fingerprint: {TUNE_SEARCH_SPACE_HASH[:12]}...")

In [14]:
# ── Tuned CV — reuses cached selectors, tuning nested per fold ─────────────────
def _tuned_cv_one_fold(train_idx, test_idx, selector, model_names, n_trials, X_pool, y):
    """One outer fold's tuning + fit work, across all active models in sequence.
    Runs in its own process under Parallel - X_pool/y/selector are read-only.
    Returns {model_name: (metrics, params, n_retained)} for this fold."""
    
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    X_train, X_test = X_pool.iloc[train_idx], X_pool.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    X_train_sel = selector.transform(X_train)
    X_test_sel  = selector.transform(X_test)
    entering_cols = selector.entering_features_

    fold_out = {}
    for name in model_names:
        if name == 'Logistic Regression':
            params = tune_lr(X_train_sel, y_train, n_trials)
            pipe   = build_tuned_lr(params, entering_cols)
        elif name == 'Random Forest':
            params = tune_rf(X_train_sel, y_train, n_trials)
            pipe   = build_tuned_rf(params, entering_cols)
        elif name == 'XGBoost':
            params = tune_xgb(X_train_sel, y_train, n_trials)
            pipe   = build_tuned_xgb(params, y_train)
        elif name == 'SVM':
            params = tune_svm(X_train_sel, y_train, n_trials)
            pipe   = build_tuned_svm(params, entering_cols)
        elif name == 'LightGBM':
            params = tune_lgb(X_train_sel, y_train, n_trials)
            pipe   = build_tuned_lgb(params)
        else:
            continue

        pipe.fit(X_train_sel, y_train)
        retained = compute_retained_features(pipe, name, entering_cols)

        y_proba = pipe.predict_proba(X_test_sel)[:, 1]
        y_pred  = pipe.predict(X_test_sel)
        y_train_proba = pipe.predict_proba(X_train_sel)[:, 1]
        train_auc = roc_auc_score(y_train, y_train_proba)

        metrics = compute_metrics(y_test, y_proba, y_pred)
        metrics['train_auc'] = train_auc
        fold_out[name] = (metrics, params, len(retained))

    return fold_out

def run_tuned_cv(X_pool, fold_splits, fold_selectors, models, base_results, n_trials, label):
    if n_trials <= 0:
        return base_results, {name: [] for name in models}, {name: [] for name in models}

    model_names   = list(models.keys())
    n_total_folds = len(fold_splits)

    # Cached on data, fold structure, model set, trial budget, pruner settings.
    tune_config = {
        'data_hash': DATA_HASH, 'label': label, 'model_names': model_names,
        'n_trials': n_trials,
        'n_splits': N_SPLITS, 'n_repeats': N_REPEATS, 'n_folds': n_total_folds,
        'random_state': RANDOM_STATE, 'pruner_startup': PRUNER._n_startup_trials,
        'pruner_warmup': PRUNER._n_warmup_steps,
        'search_space_hash': TUNE_SEARCH_SPACE_HASH,
    }
    tune_cache_path = f'{OUTPUT_DIR}/tuned_cv_cache_{label}.pkl'

    if os.path.exists(tune_cache_path):
        with open(tune_cache_path, 'rb') as f:
            cache = pickle.load(f)
        if cache['config'] == tune_config:
            print(f"Loaded cached tuned CV from: {tune_cache_path}")
            print(f"  {n_total_folds} folds x {len(model_names)} models, config matches - skipping tuning")
            tuned_results, fold_best_params, n_retained = (
                cache['tuned_results'], cache['fold_best_params'], cache['n_retained']
            )
            print(f"{'TUNING IMPACT':^60}")
            print(f"  {'Model':<23} {'Base AUC':>10} {'Tuned AUC':>11} {'Delta':>8}")
            for name in models:
                if not base_results.get(name):
                    continue
                base_auc  = np.mean([m['auc'] for m in base_results[name]])
                tuned_auc = np.mean([m['auc'] for m in tuned_results[name]])
                delta     = tuned_auc - base_auc
                delta_str = f"+{delta:.4f}" if delta >= 0 else f"{delta:.4f}"
                print(f"  {name:<23} {base_auc:>10.4f} {tuned_auc:>11.4f} {delta_str:>8}")
            return tuned_results, fold_best_params, n_retained

    tuned_results    = {name: [] for name in models}
    fold_best_params = {name: [] for name in models}
    n_retained       = {name: [] for name in models}

    print(f"Running tuned CV ({label}): {n_total_folds} folds x {len(model_names)} models "
          f"on {N_JOBS} workers, {n_trials} trials/model/fold")

    fold_pairs = list(zip(fold_splits, fold_selectors))
    fold_outputs = list(tqdm(
        Parallel(n_jobs=N_JOBS, return_as='generator')(
            delayed(_tuned_cv_one_fold)(train_idx, test_idx, selector, model_names, n_trials, X_pool, y)
            for (train_idx, test_idx), selector in fold_pairs
        ),
        total=len(fold_pairs), desc=f"{label} tuned CV", mininterval=5.0
    ))

    for fold_out in fold_outputs:
        for name, (metrics, params, n_ret) in fold_out.items():
            tuned_results[name].append(metrics)
            fold_best_params[name].append(params)
            n_retained[name].append(n_ret)

    print(f"\nTuned CV complete ({label}): {N_SPLITS*N_REPEATS} folds x {len(models)} models")

    print(f"{'TUNING IMPACT':^60}")
    print(f"  {'Model':<23} {'Base AUC':>10} {'Tuned AUC':>11} {'Delta':>8}")
    for name in models:
        if not base_results.get(name):
            continue
        base_auc  = np.mean([m['auc'] for m in base_results[name]])
        tuned_auc = np.mean([m['auc'] for m in tuned_results[name]])
        delta     = tuned_auc - base_auc
        delta_str = f"+{delta:.4f}" if delta >= 0 else f"{delta:.4f}"
        print(f"  {name:<23} {base_auc:>10.4f} {tuned_auc:>11.4f} {delta_str:>8}")

    with open(f'{OUTPUT_DIR}/fold_best_params_{label}.json', 'w') as f:
        json.dump(fold_best_params, f, indent=2)
    print(f"Saved: fold_best_params_{label}.json")

    with open(tune_cache_path, 'wb') as f:
        pickle.dump({
            'tuned_results': tuned_results, 'fold_best_params': fold_best_params,
            'n_retained': n_retained, 'config': tune_config,
        }, f)
    print(f"Cached to: {tune_cache_path}")

    return tuned_results, fold_best_params, n_retained

primary_tuned_results, primary_fold_best_params, primary_n_retained = run_tuned_cv(
    X_primary, primary_fold_splits, primary_fold_selectors, BASE_MODELS,
    primary_base_results, N_TRIALS, 'primary'
)
primary_results = primary_tuned_results

print("\nTrain vs Test AUC Gap (overfitting check) - primary:")
print(f"  {'Model':<23} {'Train AUC':>10} {'Test AUC':>10} {'Gap':>8} {'Status':>10}")
for name, fold_metrics in primary_results.items():
    if not fold_metrics:
        continue
    test_auc  = np.mean([m['auc']       for m in fold_metrics])
    train_auc = np.mean([m['train_auc'] for m in fold_metrics])
    gap = train_auc - test_auc
    status = 'OVERFIT' if gap > 0.1 else 'UNDERFIT' if test_auc < 0.65 else 'OK'
    print(f"  {name:<23} {train_auc:>10.3f} {test_auc:>10.3f} {gap:>8.3f} {status:>10}")

rows = []
for name, fold_metrics in primary_results.items():
    if not fold_metrics:
        continue
    row = {'Model': name}
    for metric in ['auc', 'sensitivity', 'specificity', 'precision', 'bal_acc', 'f1', 'mcc', 'brier']:
        vals = [m[metric] for m in fold_metrics]
        row[f'{metric}_mean'] = round(np.mean(vals), 3)
        row[f'{metric}_std']  = round(np.std(vals), 3)
    row['mean_retained'] = round(np.mean(primary_n_retained[name]), 1) if primary_n_retained.get(name) else None
    rows.append(row)

pd.DataFrame(rows).set_index('Model').to_csv(f'{OUTPUT_DIR}/cv_results_primary_tuned.csv')
print(f"\nSaved: cv_results_primary_tuned.csv")

## 6. Residual Pipeline (Entering)

Same criteria and fold splits as primary, run independently. Residual is not fit standalone, so it has no retained set of its own.

In [15]:
X_residual = X_all[RESIDUAL_VARS].copy()

if RUN_FS_GRID_SEARCH:
    # alpha here is an inert placeholder - FeatureSelector.__init__ requires it,
    # but _fit_fold_for_grid never applies it; the real sweep happens per
    # combination inside _bootstrap_for_alpha via FS_GRID['alpha'].
    _fs_base_config = {'miss_threshold': 60, 'nzv_threshold': 0.95, 'alpha': 0.05,
                        'n_bootstrap': 100, 'random_state': RANDOM_STATE}
    residual_fs_grid = run_fs_grid_search(
        X_residual, y, list(cv.split(X_residual, y)), [], {},
        _fs_base_config, 'residual'
    )

RESIDUAL_CACHE_PATH = f'{OUTPUT_DIR}/feature_selection_cache_residual.pkl'
RESIDUAL_CONFIG = {
    'miss_threshold': 60, 'nzv_threshold': 0.95, 'alpha': 0.1, 'n_bootstrap': 100,
    'top_k': 25, 'stability_threshold': 0.5, 'consensus_frac': 2/3,
    'random_state': RANDOM_STATE, 'use_bootstrap': USE_BOOTSTRAP,
    'n_splits': N_SPLITS, 'n_repeats': N_REPEATS,
    'var_set': 'residual', 'data_hash': DATA_HASH,
}

if RUN_FS_GRID_SEARCH:
    _winner = residual_fs_grid.iloc[0]
    RESIDUAL_CONFIG['top_k']               = int(_winner['top_k'])
    RESIDUAL_CONFIG['stability_threshold'] = float(_winner['stability_threshold'])
    RESIDUAL_CONFIG['consensus_frac']      = float(_winner['consensus_frac'])
    RESIDUAL_CONFIG['alpha']               = float(_winner['alpha'])
    print(f"Applied grid winner to RESIDUAL_CONFIG: top_k={RESIDUAL_CONFIG['top_k']}, "
          f"stability_threshold={RESIDUAL_CONFIG['stability_threshold']}, "
          f"consensus_frac={RESIDUAL_CONFIG['consensus_frac']:.3f}, alpha={RESIDUAL_CONFIG['alpha']}")

print("Running residual-only feature selection...")
residual_fold_splits, residual_fold_selectors = run_or_load_selection(
    X_residual, RESIDUAL_CACHE_PATH, RESIDUAL_CONFIG, onehot_vars=[], onehot_ref={}
)

residual_fold_entering = [s.entering_features_ for s in residual_fold_selectors]
mean_entering_r = np.mean([len(e) for e in residual_fold_entering])
print(f"\nResidual-only selection ready: {len(residual_fold_splits)} folds")
print(f"  Mean entering features/fold: {mean_entering_r:.1f} / {len(RESIDUAL_VARS)} candidates")

In [16]:
# ── Entering summary — primary and residual side by side ──────────────────────
entering_summary = pd.DataFrame({
    'Variable Pool': ['Primary', 'Residual'],
    'Candidates':    [len(PRIMARY_VARS), len(RESIDUAL_VARS)],
    'Mean Entering': [round(mean_entering, 1), round(mean_entering_r, 1)],
}).set_index('Variable Pool')

print(entering_summary.to_string())
entering_summary.to_csv(f'{OUTPUT_DIR}/entering_summary.csv')
print(f"\nSaved: entering_summary.csv")

## 7. Combined Pipeline (Primary Entering + Residual Entering -> Fit -> Retained)

Combined entering = primary entering + residual entering, fit fresh per fold. `cv.split()` depends only on `y` and row count, so primary/residual fold splits are identical and safe to pair.

In [17]:
splits_match = all(
    np.array_equal(p[0], r[0]) and np.array_equal(p[1], r[1])
    for p, r in zip(primary_fold_splits, residual_fold_splits)
)
print(f"Primary and residual fold splits identical: {splits_match}")
assert splits_match, "Fold splits diverged - cannot safely combine entering sets per fold."

combined_fold_splits = primary_fold_splits
X_combined = pd.concat([X_primary, X_residual], axis=1)

combined_fold_entering = [
    list(p_sel.entering_features_) + list(r_sel.entering_features_)
    for p_sel, r_sel in zip(primary_fold_selectors, residual_fold_selectors)
]
mean_entering_combined = np.mean([len(c) for c in combined_fold_entering])
print(f"Combined entering features/fold: mean = {mean_entering_combined:.1f}")
print(f"  (primary mean entering = {mean_entering:.1f}, residual mean entering = {mean_entering_r:.1f})")

In [18]:
# ── Combined selector — concatenates primary and residual transform output ────
class CombinedSelector:
    """transform() concatenates the primary selector's and residual selector's
    transform output. No selection recomputation - both halves were already
    fit on this fold's training data independently in Sections 4 and 6."""
    def __init__(self, primary_selector, residual_selector):
        self.primary_selector  = primary_selector
        self.residual_selector = residual_selector
        self.entering_features_ = (list(primary_selector.entering_features_) +
                                    list(residual_selector.entering_features_))

    def transform(self, X):
        X_p = self.primary_selector.transform(X[PRIMARY_VARS])
        X_r = self.residual_selector.transform(X[RESIDUAL_VARS])
        return pd.concat([X_p, X_r], axis=1)

combined_fold_selectors = [
    CombinedSelector(p_sel, r_sel)
    for p_sel, r_sel in zip(primary_fold_selectors, residual_fold_selectors)
]

print(f"Combined selectors ready: {len(combined_fold_selectors)} folds")

In [19]:
combined_base_results, combined_n_entering, combined_n_retained_base = run_base_cv(
    X_combined, combined_fold_splits, combined_fold_selectors, BASE_MODELS, 'combined'
)
print(f"\nCombined base CV complete: {N_SPLITS*N_REPEATS} folds")

## 8. Hyperparameter Tuning & Tuned CV - Combined

In [20]:
combined_tuned_results, combined_fold_best_params, combined_n_retained = run_tuned_cv(
    X_combined, combined_fold_splits, combined_fold_selectors, BASE_MODELS,
    combined_base_results, N_TRIALS, 'combined'
)
combined_results = combined_tuned_results

print("\nTrain vs Test AUC Gap (overfitting check) - combined:")
print(f"  {'Model':<23} {'Train AUC':>10} {'Test AUC':>10} {'Gap':>8} {'Status':>10}")
for name, fold_metrics in combined_results.items():
    if not fold_metrics:
        continue
    test_auc  = np.mean([m['auc']       for m in fold_metrics])
    train_auc = np.mean([m['train_auc'] for m in fold_metrics])
    gap = train_auc - test_auc
    status = 'OVERFIT' if gap > 0.1 else 'UNDERFIT' if test_auc < 0.65 else 'OK'
    print(f"  {name:<23} {train_auc:>10.3f} {test_auc:>10.3f} {gap:>8.3f} {status:>10}")

rows = []
for name, fold_metrics in combined_results.items():
    if not fold_metrics:
        continue
    row = {'Model': name}
    for metric in ['auc', 'sensitivity', 'specificity', 'precision', 'bal_acc', 'f1', 'mcc', 'brier']:
        vals = [m[metric] for m in fold_metrics]
        row[f'{metric}_mean'] = round(np.mean(vals), 3)
        row[f'{metric}_std']  = round(np.std(vals), 3)
    row['mean_retained'] = round(np.mean(combined_n_retained[name]), 1) if combined_n_retained.get(name) else None
    rows.append(row)

pd.DataFrame(rows).set_index('Model').to_csv(f'{OUTPUT_DIR}/cv_results_combined_tuned.csv')
print(f"\nSaved: cv_results_combined_tuned.csv")

## 9. Primary vs Combined Comparison

Does adding residual candidates change performance? Metrics and retained counts side by side, saved as CSV and figure.

In [21]:
METRICS = ['auc', 'sensitivity', 'specificity', 'precision', 'bal_acc', 'f1', 'mcc', 'brier']
METRIC_LABELS = {
    'auc':         'AUC-ROC',
    'sensitivity': 'Sensitivity',
    'specificity': 'Specificity',
    'precision':   'Precision',
    'bal_acc':     'Balanced Accuracy',
    'f1':          'F1 Score',
    'mcc':         'MCC',
    'brier':       'Brier Score',
}

comparison_rows = []
for name in BASE_MODELS:
    if not primary_results.get(name) or not combined_results.get(name):
        continue
    row = {'Model': name}
    for metric in METRICS:
        p_vals = [m[metric] for m in primary_results[name]]
        c_vals = [m[metric] for m in combined_results[name]]
        row[f'{metric}_primary_mean']  = round(np.mean(p_vals), 3)
        row[f'{metric}_primary_std']   = round(np.std(p_vals), 3)
        row[f'{metric}_combined_mean'] = round(np.mean(c_vals), 3)
        row[f'{metric}_combined_std']  = round(np.std(c_vals), 3)
        row[f'{metric}_delta']         = round(row[f'{metric}_combined_mean'] - row[f'{metric}_primary_mean'], 3)
    p_ret_vals = primary_n_retained.get(name) or primary_n_retained_base.get(name, [0])
    c_ret_vals = combined_n_retained.get(name) or combined_n_retained_base.get(name, [0])
    row['retained_primary_mean']  = round(np.mean(p_ret_vals), 1)
    row['retained_combined_mean'] = round(np.mean(c_ret_vals), 1)
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index('Model')
comparison_df.to_csv(f'{OUTPUT_DIR}/primary_vs_combined_comparison.csv')
print(f"Saved: primary_vs_combined_comparison.csv")

display_rows = []
for name in comparison_df.index:
    row = {'Model': name}
    for metric in METRICS:
        p = f"{comparison_df.loc[name, f'{metric}_primary_mean']:.3f} ± {comparison_df.loc[name, f'{metric}_primary_std']:.3f}"
        c = f"{comparison_df.loc[name, f'{metric}_combined_mean']:.3f} ± {comparison_df.loc[name, f'{metric}_combined_std']:.3f}"
        d = comparison_df.loc[name, f'{metric}_delta']
        sign = '+' if d >= 0 else ''
        row[f'{METRIC_LABELS[metric]} (Primary)']  = p
        row[f'{METRIC_LABELS[metric]} (Combined)'] = c
        row[f'{METRIC_LABELS[metric]} (Delta)']    = f"{sign}{d:.3f}"
    row['Retained (Primary)']  = comparison_df.loc[name, 'retained_primary_mean']
    row['Retained (Combined)'] = comparison_df.loc[name, 'retained_combined_mean']
    display_rows.append(row)

display(pd.DataFrame(display_rows).set_index('Model'))

In [22]:
# ── Figure: AUC comparison, primary vs combined, all models ──────────────────
model_names = [n for n in BASE_MODELS if n in comparison_df.index]
x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(max(8, len(model_names)*2), 5))

primary_aucs  = [comparison_df.loc[n, 'auc_primary_mean']  for n in model_names]
primary_stds  = [comparison_df.loc[n, 'auc_primary_std']   for n in model_names]
combined_aucs = [comparison_df.loc[n, 'auc_combined_mean'] for n in model_names]
combined_stds = [comparison_df.loc[n, 'auc_combined_std']  for n in model_names]

ax.bar(x - width/2, primary_aucs,  width, yerr=primary_stds,
       label='Primary only', color=C_OSS, alpha=0.85, edgecolor='white', capsize=4)
ax.bar(x + width/2, combined_aucs, width, yerr=combined_stds,
       label='Primary + Residual', color=C_LLAMA, alpha=0.85, edgecolor='white', capsize=4)

ax.axhline(0.5, color=C_GREY, linestyle='--', linewidth=1, alpha=0.4)
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylabel('AUC-ROC')
ax.set_ylim(0.4, 1.05)
ax.set_title('AUC-ROC - Primary Only vs Primary + Residual (Combined)', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
save_fig(f'{OUTPUT_DIR}/fig01_primary_vs_combined_auc')
plt.show()
print("Saved: fig01_primary_vs_combined_auc")

In [23]:
# ── Figure: all metrics, primary vs combined ──────────────────────────────────
metrics_to_plot = ['auc', 'sensitivity', 'specificity', 'precision', 'bal_acc', 'f1', 'mcc']
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(len(metrics_to_plot)*3.2, 5))

for ax, metric in zip(axes, metrics_to_plot):
    p_vals = [comparison_df.loc[n, f'{metric}_primary_mean']  for n in model_names]
    c_vals = [comparison_df.loc[n, f'{metric}_combined_mean'] for n in model_names]
    xx = np.arange(len(model_names))
    ax.bar(xx - width/2, p_vals, width, color=C_OSS,   alpha=0.85, edgecolor='white', label='Primary only')
    ax.bar(xx + width/2, c_vals, width, color=C_LLAMA, alpha=0.85, edgecolor='white', label='Primary + Residual')
    ax.set_title(METRIC_LABELS[metric], fontweight='bold', fontsize=10)
    ax.set_xticks(xx)
    ax.set_xticklabels(model_names, rotation=45, fontsize=8)
    ax.set_ylim(0, 1.1)
    ax.axhline(0.5, color=C_GREY, linestyle='--', linewidth=0.8, alpha=0.4)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.04),
           ncol=2, fontsize=10, frameon=False)
fig.suptitle('Primary Only vs Primary + Residual - All Metrics',
             fontsize=13, fontweight='bold', y=1.12)
plt.tight_layout()
save_fig(f'{OUTPUT_DIR}/fig02_primary_vs_combined_all_metrics')
plt.show()
print("Saved: fig02_primary_vs_combined_all_metrics")

In [24]:
# ── Figure: retained feature count, primary vs combined ───────────────────────
fig, ax = plt.subplots(figsize=(max(8, len(model_names)*2), 4.5))
p_ret = [comparison_df.loc[n, 'retained_primary_mean']  for n in model_names]
c_ret = [comparison_df.loc[n, 'retained_combined_mean'] for n in model_names]

ax.bar(x - width/2, p_ret, width, label='Primary only', color=C_OSS, alpha=0.85, edgecolor='white')
ax.bar(x + width/2, c_ret, width, label='Primary + Residual', color=C_LLAMA, alpha=0.85, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylabel('Mean retained features per fold')
ax.set_title('Retained Feature Count - Primary Only vs Combined', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
save_fig(f'{OUTPUT_DIR}/fig03_retained_feature_count')
plt.show()
print("Saved: fig03_retained_feature_count")

## 10. Ensemble (Primary and Combined)

In [25]:
def _build_tuned_estimator(name, params, entering_cols, y_train):
    if name == 'Logistic Regression': return build_tuned_lr(params, entering_cols)
    if name == 'Random Forest':       return build_tuned_rf(params, entering_cols)
    if name == 'XGBoost':             return build_tuned_xgb(params, y_train)
    if name == 'SVM':                 return build_tuned_svm(params, entering_cols)
    if name == 'LightGBM':            return build_tuned_lgb(params)
    raise ValueError(f"Unknown model: {name}")

def _ensemble_cv_one_fold(fold_idx, train_idx, test_idx, selector, model_names, fold_best_params, X_pool, y):
    """One outer fold's ensemble fit, using each model's tuned params for this
    fold (same params run_tuned_cv already found). Vote weights come from an
    inner CV on the training fold only, so no test-fold leakage."""
    X_train, X_test = X_pool.iloc[train_idx], X_pool.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    X_train_sel = selector.transform(X_train)
    X_test_sel  = selector.transform(X_test)
    entering_cols = selector.entering_features_

    inner_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    estimators, weights = [], []
    for name in model_names:
        params = fold_best_params[name][fold_idx]
        est = _build_tuned_estimator(name, params, entering_cols, y_train)
        weights.append(np.mean(cross_val_score(est, X_train_sel, y_train, cv=inner_cv, scoring='roc_auc')))
        estimators.append((name.lower()[:3], est))

    voting_clf = VotingClassifier(estimators=estimators, voting='soft', weights=weights)
    voting_clf.fit(X_train_sel, y_train)

    avg_proba = voting_clf.predict_proba(X_test_sel)[:, 1]
    avg_pred  = voting_clf.predict(X_test_sel)
    y_train_proba = voting_clf.predict_proba(X_train_sel)[:, 1]
    train_auc = roc_auc_score(y_train, y_train_proba)

    metrics = compute_metrics(y_test, avg_proba, avg_pred)
    metrics['train_auc'] = train_auc
    return metrics

def run_ensemble_cv(X_pool, fold_splits, fold_selectors, fold_best_params, label):
    print(f"Running ensemble ({label}): {len(fold_splits)} folds on {N_JOBS} workers")
    # If tuning was skipped (N_TRIALS=0), fold_best_params values are empty
    # lists - fall back to base models rather than indexing into nothing.
    model_names = [name for name, plist in fold_best_params.items() if plist]
    if not model_names:
        model_names = list(BASE_MODELS.keys())
        fold_best_params = {name: [{} for _ in fold_splits] for name in model_names}

    fold_pairs = list(zip(fold_splits, fold_selectors))
    ensemble_fold_metrics = list(tqdm(
        Parallel(n_jobs=N_JOBS, return_as='generator')(
            delayed(_ensemble_cv_one_fold)(i, train_idx, test_idx, selector, model_names, fold_best_params, X_pool, y)
            for i, ((train_idx, test_idx), selector) in enumerate(fold_pairs)
        ),
        total=len(fold_pairs), desc=f"{label} ensemble", mininterval=5.0
    ))

    print(f"Ensemble ({label}) results:")
    for metric in METRICS:
        vals = [m[metric] for m in ensemble_fold_metrics]
        print(f"  {METRIC_LABELS[metric]:<22}: {np.mean(vals):.3f} ± {np.std(vals):.3f}")

    return ensemble_fold_metrics

primary_results['Ensemble']  = run_ensemble_cv(X_primary,  primary_fold_splits,  primary_fold_selectors,  primary_fold_best_params,  'primary')
combined_results['Ensemble'] = run_ensemble_cv(X_combined, combined_fold_splits, combined_fold_selectors, combined_fold_best_params, 'combined')

In [26]:
# ── Updated comparison including Ensemble ──────────────────────────────────────
comparison_rows_full = []
for name in list(BASE_MODELS) + ['Ensemble']:
    if not primary_results.get(name) or not combined_results.get(name):
        continue
    row = {'Model': name}
    for metric in METRICS:
        p_vals = [m[metric] for m in primary_results[name]]
        c_vals = [m[metric] for m in combined_results[name]]
        row[f'{metric}_primary_mean']  = round(np.mean(p_vals), 3)
        row[f'{metric}_primary_std']   = round(np.std(p_vals), 3)
        row[f'{metric}_combined_mean'] = round(np.mean(c_vals), 3)
        row[f'{metric}_combined_std']  = round(np.std(c_vals), 3)
        row[f'{metric}_delta']         = round(row[f'{metric}_combined_mean'] - row[f'{metric}_primary_mean'], 3)
    comparison_rows_full.append(row)

comparison_full_df = pd.DataFrame(comparison_rows_full).set_index('Model')
comparison_full_df.to_csv(f'{OUTPUT_DIR}/primary_vs_combined_comparison_final.csv')
print(f"Saved: primary_vs_combined_comparison_final.csv")

display_rows_full = []
for name in comparison_full_df.index:
    row = {'Model': name}
    for metric in METRICS:
        p = f"{comparison_full_df.loc[name, f'{metric}_primary_mean']:.3f} ± {comparison_full_df.loc[name, f'{metric}_primary_std']:.3f}"
        c = f"{comparison_full_df.loc[name, f'{metric}_combined_mean']:.3f} ± {comparison_full_df.loc[name, f'{metric}_combined_std']:.3f}"
        d = comparison_full_df.loc[name, f'{metric}_delta']
        sign = '+' if d >= 0 else ''
        row[f'{METRIC_LABELS[metric]} (Primary)']  = p
        row[f'{METRIC_LABELS[metric]} (Combined)'] = c
        row[f'{METRIC_LABELS[metric]} (Delta)']    = f"{sign}{d:.3f}"
    display_rows_full.append(row)

display(pd.DataFrame(display_rows_full).set_index('Model'))

## 11. Final Feature Set & Fit

Nested selection means different folds can have different entering sets, so there's no single feature list for SHAP/coefficients. Standard practice: report performance from the outer loop, then build one final entering set and fit for interpretation, for both pipelines.

Hyperparameters use the median of each pipeline's per-fold tuned values. Retained features computed the same way as everywhere else, on the full-dataset fit.

In [27]:
# ── Final entering selection — single fit on full dataset, both pipelines ─────
_FS_KWARGS = ('miss_threshold', 'nzv_threshold', 'alpha', 'n_bootstrap',
              'top_k', 'stability_threshold', 'consensus_frac', 'use_bootstrap')

final_primary_selector = FeatureSelector(
    **{k: PRIMARY_CONFIG[k] for k in _FS_KWARGS},
    random_state=RANDOM_STATE, onehot_vars=ONEHOT_VARS, onehot_ref=ONEHOT_REF,
)
final_primary_selector.fit(X_primary, y)
FEATURE_COLS_PRIMARY = final_primary_selector.entering_features_

final_residual_selector = FeatureSelector(
    **{k: RESIDUAL_CONFIG[k] for k in _FS_KWARGS},
    random_state=RANDOM_STATE, onehot_vars=[], onehot_ref={},
)
final_residual_selector.fit(X_residual, y)
FEATURE_COLS_RESIDUAL = final_residual_selector.entering_features_

FEATURE_COLS_COMBINED = list(FEATURE_COLS_PRIMARY) + list(FEATURE_COLS_RESIDUAL)
final_combined_selector = CombinedSelector(final_primary_selector, final_residual_selector)

print(f"Final PRIMARY entering set:  {len(FEATURE_COLS_PRIMARY)} features")
for c in FEATURE_COLS_PRIMARY:
    print(f"  {c}")
print(f"\nFinal RESIDUAL entering set: {len(FEATURE_COLS_RESIDUAL)} features")
for c in FEATURE_COLS_RESIDUAL:
    print(f"  {c}")
print(f"\nFinal COMBINED entering set: {len(FEATURE_COLS_COMBINED)} features")

with open(f'{OUTPUT_DIR}/feature_list_primary_entering.txt', 'w') as f:
    for col in FEATURE_COLS_PRIMARY:
        f.write(col + '\n')
with open(f'{OUTPUT_DIR}/feature_list_combined_entering.txt', 'w') as f:
    for col in FEATURE_COLS_COMBINED:
        f.write(col + '\n')
print(f"\nSaved: feature_list_primary_entering.txt, feature_list_combined_entering.txt")

In [28]:
# ── Selection frequency across outer folds — how stable is each entering set? ──
def fold_selection_frequency(fold_selectors, final_features):
    return pd.Series({
        c: sum(c in set(sel.entering_features_) for sel in fold_selectors) / len(fold_selectors)
        for c in final_features
    }).sort_values(ascending=False)

primary_freq  = fold_selection_frequency(primary_fold_selectors,  FEATURE_COLS_PRIMARY)
residual_freq = fold_selection_frequency(residual_fold_selectors, FEATURE_COLS_RESIDUAL)

_n_outer_folds = N_SPLITS * N_REPEATS
print(f"Final PRIMARY entering set - selection frequency across {_n_outer_folds} outer-fold fits:")
for c, f in primary_freq.items():
    print(f"  {c:42s}  {f:.2f}")
print(f"\nFinal RESIDUAL entering set - selection frequency across {_n_outer_folds} outer-fold fits:")
for c, f in residual_freq.items():
    print(f"  {c:42s}  {f:.2f}")

primary_freq.to_csv(f'{OUTPUT_DIR}/fold_selection_frequency_primary.csv')
residual_freq.to_csv(f'{OUTPUT_DIR}/fold_selection_frequency_residual.csv')
print(f"\nSaved: fold_selection_frequency_primary.csv, fold_selection_frequency_residual.csv")

In [29]:
# ── Median tuned hyperparameters across outer folds, both pipelines ────────────
primary_lr_params  = median_params(primary_fold_best_params.get('Logistic Regression', []))
primary_rf_params  = median_params(primary_fold_best_params.get('Random Forest', []),
                                    int_keys=('n_estimators', 'max_depth', 'min_samples_leaf', 'min_samples_split'))
primary_xgb_params = median_params(primary_fold_best_params.get('XGBoost', []),
                                    int_keys=('max_depth', 'n_estimators', 'min_child_weight'))
primary_svm_params = median_params(primary_fold_best_params.get('SVM', []))
primary_lgb_params = median_params(primary_fold_best_params.get('LightGBM', []),
                                    int_keys=('num_leaves', 'max_depth', 'n_estimators', 'min_child_samples'))

combined_lr_params  = median_params(combined_fold_best_params.get('Logistic Regression', []))
combined_rf_params  = median_params(combined_fold_best_params.get('Random Forest', []),
                                     int_keys=('n_estimators', 'max_depth', 'min_samples_leaf', 'min_samples_split'))
combined_xgb_params = median_params(combined_fold_best_params.get('XGBoost', []),
                                     int_keys=('max_depth', 'n_estimators', 'min_child_weight'))
combined_svm_params = median_params(combined_fold_best_params.get('SVM', []))
combined_lgb_params = median_params(combined_fold_best_params.get('LightGBM', []),
                                     int_keys=('num_leaves', 'max_depth', 'n_estimators', 'min_child_samples'))

print("Median hyperparameters - primary:")
print(f"  LR:  {primary_lr_params}")
print(f"  RF:  {primary_rf_params}")
print(f"  XGB: {primary_xgb_params}")
if USE_SVM:      print(f"  SVM: {primary_svm_params}")
if USE_LIGHTGBM: print(f"  LGB: {primary_lgb_params}")
print("\nMedian hyperparameters - combined:")
print(f"  LR:  {combined_lr_params}")
print(f"  RF:  {combined_rf_params}")
print(f"  XGB: {combined_xgb_params}")
if USE_SVM:      print(f"  SVM: {combined_svm_params}")
if USE_LIGHTGBM: print(f"  LGB: {combined_lgb_params}")

In [30]:
# ── Rebuild final pipelines, fit on full entering set, compute final retained ──
# transform() handles one-hot dummy column names; raw column slicing would not.
X_final_primary  = final_primary_selector.transform(X_primary)
X_final_combined = final_combined_selector.transform(X_combined)

def build_final_models(lr_params, rf_params, xgb_params, svm_params, lgb_params, feature_cols, y_full):
    models = {
        'Logistic Regression': build_tuned_lr(lr_params, feature_cols),
        'Random Forest':       build_tuned_rf(rf_params, feature_cols),
    }
    if USE_SVM:      models['SVM']      = build_tuned_svm(svm_params, feature_cols)
    if USE_LIGHTGBM: models['LightGBM'] = build_tuned_lgb(lgb_params)
    if USE_XGBOOST:  models['XGBoost']  = build_tuned_xgb(xgb_params, y_full)
    return models

MODELS_PRIMARY  = build_final_models(primary_lr_params,  primary_rf_params,  primary_xgb_params,
                                      primary_svm_params, primary_lgb_params, FEATURE_COLS_PRIMARY,  y)
MODELS_COMBINED = build_final_models(combined_lr_params, combined_rf_params, combined_xgb_params,
                                      combined_svm_params, combined_lgb_params, FEATURE_COLS_COMBINED, y)

fitted_primary, final_retained_primary = {}, {}
for name, pipe in MODELS_PRIMARY.items():
    pipe.fit(X_final_primary, y)
    fitted_primary[name] = pipe
    final_retained_primary[name] = compute_retained_features(pipe, name, FEATURE_COLS_PRIMARY)
    print(f"Fitted (final, primary): {name} | retained = {len(final_retained_primary[name])}")

fitted_combined, final_retained_combined = {}, {}
for name, pipe in MODELS_COMBINED.items():
    pipe.fit(X_final_combined, y)
    fitted_combined[name] = pipe
    final_retained_combined[name] = compute_retained_features(pipe, name, FEATURE_COLS_COMBINED)
    print(f"Fitted (final, combined): {name} | retained = {len(final_retained_combined[name])}")

In [31]:
# ── Final retained feature lists — saved for direct paper use ──────────────────
for name, feats in final_retained_primary.items():
    fname = f"retained_primary_{name.lower().replace(' ', '_')}.txt"
    with open(f'{OUTPUT_DIR}/{fname}', 'w') as f:
        for c in feats:
            f.write(c + '\n')
    print(f"Saved: {fname} ({len(feats)} features)")

for name, feats in final_retained_combined.items():
    fname = f"retained_combined_{name.lower().replace(' ', '_')}.txt"
    with open(f'{OUTPUT_DIR}/{fname}', 'w') as f:
        for c in feats:
            f.write(c + '\n')
    print(f"Saved: {fname} ({len(feats)} features)")

## 12. Feature Importance & SHAP

In [32]:
def get_tree_models(models):
    tree_models = {}
    if 'Random Forest' in models:
        tree_models['Random Forest'] = models['Random Forest']
    if USE_LIGHTGBM and 'LightGBM' in models:
        tree_models['LightGBM'] = models['LightGBM']
    if USE_XGBOOST and 'XGBoost' in models:
        tree_models['XGBoost'] = models['XGBoost']
    return tree_models

TREE_MODELS_PRIMARY  = get_tree_models(fitted_primary)
TREE_MODELS_COMBINED = get_tree_models(fitted_combined)

print(f"Tree models for SHAP (primary):  {list(TREE_MODELS_PRIMARY.keys())}")
print(f"Tree models for SHAP (combined): {list(TREE_MODELS_COMBINED.keys())}")
print("LR uses coefficient-based importance for both")

In [33]:
# ── LR Coefficients — primary and combined ─────────────────────────────────────
fig_num = 4

for label, fitted_models, feature_cols in [
    ('Primary',  fitted_primary,  FEATURE_COLS_PRIMARY),
    ('Combined', fitted_combined, FEATURE_COLS_COMBINED),
]:
    lr_model = fitted_models['Logistic Regression']['model']
    lr_coefs = pd.Series(lr_model.coef_[0], index=feature_cols).sort_values()

    fig, ax = plt.subplots(figsize=(7, max(5, len(feature_cols)*0.3)))
    colors_coef = [C_OSS if v > 0 else C_QWEN for v in lr_coefs.values]
    ax.barh(lr_coefs.index, lr_coefs.values, color=colors_coef, alpha=0.85, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Logistic Regression Coefficients - {label}\n(ElasticNet regularized)',
                 fontweight='bold')
    ax.set_xlabel('Coefficient value')
    ax.tick_params(labelsize=8)
    legend_handles = [
        Patch(facecolor=C_OSS,  alpha=0.85, label='Increases ADNC odds'),
        Patch(facecolor=C_QWEN, alpha=0.85, label='Decreases ADNC odds'),
    ]
    ax.legend(handles=legend_handles, fontsize=8, loc='lower right')

    plt.tight_layout()
    save_fig(f'{OUTPUT_DIR}/fig{fig_num:02d}_lr_coefficients_{label.lower()}')
    plt.show()
    print(f"Saved: fig{fig_num:02d}_lr_coefficients_{label.lower()}")
    fig_num += 1

In [34]:
# ── SHAP summary plots — primary and combined, all tree models ────────────────
def shap_summary(name, pipe, X_final, feature_cols):
    if name == 'Random Forest':
        model_obj  = pipe['model']
        X_shap_imp = pipe['imputer'].transform(X_final)
        X_shap     = pd.DataFrame(X_shap_imp, columns=feature_cols)
    else:
        model_obj = pipe
        X_shap    = X_final.copy()

    explainer   = shap.TreeExplainer(model_obj)
    shap_values = explainer.shap_values(X_shap)

    shap_vals = np.array(shap_values)
    if shap_vals.ndim == 3:
        shap_vals = shap_vals[:, :, 1]
    elif isinstance(shap_values, list):
        shap_vals = np.array(shap_values[1])
    else:
        shap_vals = shap_values

    return shap_vals, X_shap

for label, tree_models, X_final, feature_cols in [
    ('Primary',  TREE_MODELS_PRIMARY,  X_final_primary,  FEATURE_COLS_PRIMARY),
    ('Combined', TREE_MODELS_COMBINED, X_final_combined, FEATURE_COLS_COMBINED),
]:
    for name, pipe in tree_models.items():
        print(f"Computing SHAP for {name} ({label}) ...")
        shap_vals, X_shap = shap_summary(name, pipe, X_final, feature_cols)

        fig, ax = plt.subplots(figsize=(9, max(5, len(feature_cols)*0.3)))
        shap.summary_plot(
            shap_vals, X_shap,
            feature_names=feature_cols,
            plot_type='bar',
            show=False,
            color=MODEL_COLORS.get(name, C_GREY),
        )
        plt.title(f'SHAP Feature Importance - {name} ({label})', fontweight='bold', fontsize=12)
        plt.tight_layout()
        fname = f'fig{fig_num:02d}_shap_{name.lower().replace(" ","_")}_{label.lower()}'
        save_fig(f'{OUTPUT_DIR}/{fname}')
        plt.show()
        print(f"Saved: {fname}")
        fig_num += 1

## 13. Error Analysis

In [35]:
inner_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

def compute_oof(models, X_final):
    oof_probas, oof_preds = {}, {}
    for name, pipe in models.items():
        print(f"Computing OOF predictions: {name} ...", end=' ')
        oof_probas[name] = cross_val_predict(pipe, X_final, y, cv=inner_cv, method='predict_proba', n_jobs=1)[:, 1]
        oof_preds[name]  = cross_val_predict(pipe, X_final, y, cv=inner_cv, method='predict', n_jobs=1)
        print("done")
    return oof_probas, oof_preds

oof_probas_primary,  oof_preds_primary  = compute_oof(fitted_primary,  X_final_primary)
oof_probas_combined, oof_preds_combined = compute_oof(fitted_combined, X_final_combined)

for label, oof_probas, oof_preds, models in [
    ('primary',  oof_probas_primary,  oof_preds_primary,  fitted_primary),
    ('combined', oof_probas_combined, oof_preds_combined, fitted_combined),
]:
    oof_df = pd.DataFrame({'report_id': df['report_id'].values, 'true_label': y.values})
    for name in models:
        oof_df[f'{name}_proba'] = oof_probas[name]
        oof_df[f'{name}_pred']  = oof_preds[name]
    oof_df.to_csv(f'{OUTPUT_DIR}/oof_predictions_{label}.csv', index=False)
    print(f"Saved: oof_predictions_{label}.csv")

In [36]:
# ── ROC Curves — primary and combined side by side ─────────────────────────────
# AUC in the legend matches the curve: computed from oof_probas directly,
# not the 50-fold repeated-CV mean (reported separately in fig01/comparison tables).
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, label, oof_probas, models in [
    (axes[0], 'Primary',  oof_probas_primary,  fitted_primary),
    (axes[1], 'Combined', oof_probas_combined, fitted_combined),
]:
    ax.plot([0,1], [0,1], 'k--', linewidth=1, alpha=0.5, label='Random')
    for name in models:
        fpr, tpr, _ = roc_curve(y, oof_probas[name])
        auc_val = roc_auc_score(y, oof_probas[name])
        ax.plot(fpr, tpr, color=MODEL_COLORS.get(name, C_GREY),
                linewidth=2, label=f'{name} (AUC={auc_val:.3f})')
    ax.set_xlabel('False Positive Rate (1 - Specificity)')
    ax.set_ylabel('True Positive Rate (Sensitivity)')
    ax.set_title(f'ROC Curves - {label}', fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)

plt.tight_layout()
save_fig(f'{OUTPUT_DIR}/fig{fig_num:02d}_roc_curves')
plt.show()
print(f"Saved: fig{fig_num:02d}_roc_curves")
fig_num += 1

In [37]:
# ── Precision-Recall Curves — primary and combined side by side ───────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
baseline = (y == 1).mean()

for ax, label, oof_probas, models in [
    (axes[0], 'Primary',  oof_probas_primary,  fitted_primary),
    (axes[1], 'Combined', oof_probas_combined, fitted_combined),
]:
    ax.axhline(baseline, color='k', linestyle='--', linewidth=1, alpha=0.5, label=f'Random (AP={baseline:.3f})')
    for name in models:
        precision, recall, _ = precision_recall_curve(y, oof_probas[name])
        ap = average_precision_score(y, oof_probas[name])
        ax.plot(recall, precision, color=MODEL_COLORS.get(name, C_GREY),
                linewidth=2, label=f'{name} (AP={ap:.3f})')
    ax.set_xlabel('Recall (Sensitivity)')
    ax.set_ylabel('Precision')
    ax.set_title(f'Precision-Recall Curves - {label}', fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)

plt.tight_layout()
save_fig(f'{OUTPUT_DIR}/fig{fig_num:02d}_pr_curves')
plt.show()
print(f"Saved: fig{fig_num:02d}_pr_curves")
fig_num += 1

In [38]:
# ── Confusion Matrices — primary and combined ─────────────────────────────────
for label, models, oof_preds in [
    ('Primary',  fitted_primary,  oof_preds_primary),
    ('Combined', fitted_combined, oof_preds_combined),
]:
    n_models = len(models)
    fig, axes = plt.subplots(1, n_models, figsize=(n_models*3.5, 4))
    if n_models == 1:
        axes = [axes]

    for ax, name in zip(axes, models):
        cm = confusion_matrix(y, oof_preds[name])
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not/Low AD', 'Int/High AD'])
        disp.plot(ax=ax, colorbar=False, cmap='Blues')
        ax.set_title(name, fontweight='bold', fontsize=10)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True' if ax == axes[0] else '')

    fig.suptitle(f'Confusion Matrices - {label} (5-fold OOF predictions)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig(f'{OUTPUT_DIR}/fig{fig_num:02d}_confusion_matrices_{label.lower()}')
    plt.show()
    print(f"Saved: fig{fig_num:02d}_confusion_matrices_{label.lower()}")
    fig_num += 1

cm_data = []
for label, models, oof_preds in [('Primary', fitted_primary, oof_preds_primary), ('Combined', fitted_combined, oof_preds_combined)]:
    for name in models:
        tn, fp, fn, tp = confusion_matrix(y, oof_preds[name]).ravel()
        cm_data.append({'Pipeline': label, 'Model': name, 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp})
pd.DataFrame(cm_data).set_index(['Pipeline', 'Model']).to_csv(f'{OUTPUT_DIR}/confusion_matrix_data.csv')
print("Saved: confusion_matrix_data.csv")

In [39]:
# ── Calibration Plot — primary and combined side by side ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))

for ax, label, oof_probas, models in [
    (axes[0], 'Primary',  oof_probas_primary,  fitted_primary),
    (axes[1], 'Combined', oof_probas_combined, fitted_combined),
]:
    ax.plot([0,1], [0,1], 'k--', label='Perfect calibration')
    for name in models:
        fraction_pos, mean_pred = calibration_curve(y, oof_probas[name], n_bins=5)
        ax.plot(mean_pred, fraction_pos, marker='o', color=MODEL_COLORS.get(name, C_GREY), label=name)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_title(f'Calibration Plot - {label}', fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
save_fig(f'{OUTPUT_DIR}/fig{fig_num:02d}_calibration')
plt.show()
print(f"Saved: fig{fig_num:02d}_calibration")
fig_num += 1

In [40]:
# ── Classification Report — primary and combined ──────────────────────────────
for label, models, oof_preds in [('Primary', fitted_primary, oof_preds_primary), ('Combined', fitted_combined, oof_preds_combined)]:
    for name in models:
        report_dict = classification_report(
            y, oof_preds[name], target_names=['Not/Low AD', 'Int/High AD'], digits=3, output_dict=True
        )
        report_df = pd.DataFrame(report_dict).T
        print(f"\n{label} - {name}")
        display(report_df.round(3))

In [41]:
# ── Identify consistently misclassified reports — primary and combined ────────
for label, models, oof_preds in [('Primary', fitted_primary, oof_preds_primary), ('Combined', fitted_combined, oof_preds_combined)]:
    error_counts = pd.Series(0, index=df.index)
    for name in models:
        misclassified = (oof_preds[name] != y).astype(int)
        error_counts += misclassified

    error_df = pd.DataFrame({
        'report_id':   df['report_id'].values,
        'true_label':  y.values,
        'error_count': error_counts.values,
    }).sort_values('error_count', ascending=False)

    print(f"\n{label} - reports misclassified by multiple models:")
    print(error_df[error_df['error_count'] >= len(models)-1].to_string(index=False))

    error_df.to_csv(f'{OUTPUT_DIR}/error_analysis_{label.lower()}.csv', index=False)
    print(f"Saved: error_analysis_{label.lower()}.csv")

    if label == 'Primary':
        error_df_primary = error_df
    else:
        error_df_combined = error_df

In [42]:
# ── Error count distribution — primary and combined ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, label, error_df, models in [
    (axes[0], 'Primary',  error_df_primary,  fitted_primary),
    (axes[1], 'Combined', error_df_combined, fitted_combined),
]:
    vc = error_df['error_count'].value_counts().sort_index()
    ax.bar(vc.index, vc.values, color=C_OSS, alpha=0.85, edgecolor='white')
    ax.set_xlabel('Number of models that misclassified the report')
    ax.set_ylabel('Number of reports')
    ax.set_title(f'Error Distribution - {label}', fontweight='bold')
    ax.set_xticks(range(len(models) + 1))

plt.tight_layout()
save_fig(f'{OUTPUT_DIR}/fig{fig_num:02d}_error_distribution')
plt.show()
print(f"Saved: fig{fig_num:02d}_error_distribution")